In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import math

from datetime import datetime
import glob
import os
import sys

import scipy
from scipy import stats
from scipy.stats import ttest_ind_from_stats, ttest_ind, chisquare, linregress, ttest_rel
import statistics
from scipy.ndimage import generic_filter
from scipy.ndimage import label, find_objects, sum as ndi_sum, generate_binary_structure, iterate_structure, binary_dilation, binary_fill_holes, labeled_comprehension

from scipy.stats import entropy
import pymannkendall as mk
# from scipy.io import loadmat
import statsmodels.api as sm
# import statsmodels.formula.api as smf

import statsmodels as sm
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.stats.proportion import proportions_ztest

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rc
from matplotlib import cm
import matplotlib.ticker as ticker
import matplotlib as mpl
from matplotlib.colors import ListedColormap, LinearSegmentedColormap, BoundaryNorm

# mpl.rcParams['hatch.linewidth'] = 2.5  # for significant hatching

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cftime

import geopandas as gpd
import geoplot as gplt
import geoplot.crs as gcrs
import regionmask
from shapely.geometry import Point, Polygon, MultiPolygon
import xesmf as xe
import tobler

# import bottleneck
# reduce warnings (less pink)
import warnings
warnings.filterwarnings('ignore')

print('Loaded Libraries')

In [ ]:
def _metaread(dir_meta,domain):
    file = "%swrfinput_%s" %(dir_meta,domain)
    data = xr.open_dataset(file)
    lat = data["XLAT"].sel(Time=0)
    lon = data["XLONG"].sel(Time=0)

    return (lat,lon,file)


def _dtify(df):

    days = df.day.values
    dts = [datetime.strptime(str(int(d)),"%Y%m%d") for d in days]
    df['day'] = dts

    return df


def load_stef_daily_var(var,sim_path,domain):

    dir_meta = "/data/modelOutput/WRF_2020/meta/meta_new/"
    lat_wrf, lon_wrf, file = _metaread(dir_meta,domain)

    ds = xr.open_mfdataset(f'{sim_path}/{var}.daily*.nc')
    ds = ds.rename({'lat2d':'south_north','lon2d':'west_east'})
    ds = ds.assign_coords({'lat' : lat_wrf})
    ds = ds.drop('lat')
    ds = ds.rename({'XLAT':'lat','XLONG':'lon'})
    ds = _dtify(ds)

    return ds

In [ ]:
%%time

pyrome = gpd.read_file('data/Pyromes/Pyromes_lon105W.shp')
pyrome.index = pyrome['PYROME'].values

# FIRED = gpd.read_file('data/FIRED_2023_105W/fired_conus_ak_2000_to_2024_daily_105W.shp')
# FIRED['date'] = [i[:10] for i in FIRED['date']]
# FIRED['year'] = [int(i[:4]) for i in FIRED['date']]
# FIRED['month'] = [int(i[5:7]) for i in FIRED['date']]
# FIRED['ig_date'] = [i[:10] for i in FIRED['ig_date']]
# FIRED['last_date'] = [i[:10] for i in FIRED['last_date']]
# FIRED = FIRED[pd.to_datetime(FIRED['last_date']) <= pd.to_datetime('2023-08-31')]

# FIRED = FIRED.to_crs('EPSG:4326')
# FIRED = FIRED[FIRED['ig_year'] >= 2001]
# # FIRED = FIRED[FIRED['year'] <= 2018]
# # FIRED = gpd.sjoin(FIRED, pyrome)

# FIRED = FIRED[[('Crop' not in FIRED['lc_name'].iloc[i]) & ('Urban' not in FIRED['lc_name'].iloc[i]) for i in range(len(FIRED))]]

# # for id in FIRED['id'].unique():
# #     FIRED.loc[FIRED['id'] == id,'geometry'] = [FIRED[(FIRED['id'] == id) & (FIRED['event_day'] == FIRED[FIRED['id'] == id]['event_day'].max())]['geometry'].values] * len(FIRED[FIRED['id'] == id])

# FIRED = tobler.area_weighted.area_join(pyrome,FIRED,'PYROME')

# # for id in FIRED['id'].unique():
# #     FIRED.loc[FIRED['id'] == id,'prv_day_tot_ar_km2'] = [FIRED[(FIRED['event_day'] < FIRED.loc[i,'event_day']) & (FIRED['id'] == id)]['dy_ar_km2'].sum() for i in FIRED.loc[FIRED['id'] == id].index]

# # FIRED.to_file('data/FIRED_weighted_spatial_join_to_pyrome/FIRED_weighted_spatial_join_to_pyrome.shp')
# # FIRED = gpd.read_file('data/FIRED_weighted_spatial_join_to_pyrome/FIRED_weighted_spatial_join_to_pyrome.shp')
# # FIRED = FIRED.rename({'prv_day_cu':'prv_day_cum_ar_km2'},axis=1)

# FIRED.to_file('data/FIRED_weight_spatial_join_to_pyrome_by_day/FIRED_weight_spatial_join_to_pyrome_by_day.shp')
FIRED = gpd.read_file('data/FIRED_weight_spatial_join_to_pyrome_by_day/FIRED_weight_spatial_join_to_pyrome_by_day.shp')

FIRED[FIRED['id'] == 66521]

In [ ]:
pyrome = gpd.read_file('data/Pyromes/Pyromes_lon105W.shp')
pyrome.index = pyrome['PYROME'].values

annual_risk_list = []
for pyrome_id in sorted(pyrome['PYROME']):
    data = FIRED[(FIRED['PYROME'] == pyrome_id)][['event_day','id','year']].reset_index()
    annual_risk_list += [len(np.unique(data[(data['event_day'] == 1) & (data['year'] < 2023)]['id']))]

pyrome['id'] = range(1,len(pyrome)+1)
pyrome['fire_count'] = annual_risk_list
pyrome['fire_count_per_year'] = np.array(annual_risk_list)/22

pyrome.head()

In [ ]:
era5_wrf_basin_dict = {}

var_list = ['e','e_s','rh','vpd','wspd','wspdmax','soil_m','SOIL_M','hdw','fwi','ffwi','mffwi','haines','kbdi',"fm1","fm10","fm100","fm1000","erc","bi","sc","ic","ros",'isi','bui','dc','ffmc','dmc']

for var in var_list:
    era5_wrf_basin_dict[var] = xr.open_dataset(f'data/daily_{var}_by_pyrome_with_ignitions.nc')
    era5_wrf_basin_dict[var] = era5_wrf_basin_dict[var].load()
    era5_wrf_basin_dict[var] = era5_wrf_basin_dict[var].sel(day = slice('2001','2023'))

# era5_wrf_basin_dict[var] = era5_wrf_basin_dict[var].sel(day = [pd.to_datetime(i).month in list(range(3,11)) for i in era5_wrf_basin_dict[var].day.values])
era5_wrf_basin_dict[var]

In [ ]:
era5_wrf_basin_anomaly_dict = {}

var_list = ['e','e_s','rh','vpd','wspd','wspdmax','soil_m','SOIL_M','hdw','fwi','ffwi','mffwi','haines','kbdi',"fm1","fm10","fm100","fm1000","erc","bi","sc","ic","ros",'isi','bui','dc','ffmc','dmc']

for var in var_list:
    era5_wrf_basin_anomaly_dict[var] = xr.open_dataset(f'data/daily_{var}_by_pyrome_with_ignitions.nc')
    era5_wrf_basin_anomaly_dict[var] = era5_wrf_basin_anomaly_dict[var].load()
    era5_wrf_basin_anomaly_dict[var] = era5_wrf_basin_anomaly_dict[var].groupby('day.dayofyear') - era5_wrf_basin_anomaly_dict[var].groupby('day.dayofyear').mean()
    era5_wrf_basin_anomaly_dict[var] = era5_wrf_basin_anomaly_dict[var].sel(day = slice('2001','2023'))

    # era5_wrf_basin_anomaly_dict[var] = era5_wrf_basin_anomaly_dict[var].sel(day = [pd.to_datetime(i).month in list(range(6,10)) for i in era5_wrf_basin_anomaly_dict[var].day.values])
era5_wrf_basin_anomaly_dict[var]


In [ ]:
era5_wrf_basin_season_dict = {}

var_list = ['e','e_s','rh','vpd','wspd','wspdmax','soil_m','SOIL_M','hdw','fwi','ffwi','mffwi','haines','kbdi',"fm1","fm10","fm100","fm1000","erc","bi","sc","ic","ros",'isi','bui','dc','ffmc','dmc']

for var in var_list:
    era5_wrf_basin_season_dict[var] = xr.open_dataset(f'data/daily_{var}_by_pyrome_with_ignitions.nc')
    era5_wrf_basin_season_dict[var] = era5_wrf_basin_season_dict[var].load()
    era5_wrf_basin_season_dict[var] = era5_wrf_basin_season_dict[var] - (era5_wrf_basin_season_dict[var].groupby('day.dayofyear') - era5_wrf_basin_season_dict[var].groupby('day.dayofyear').mean())
    era5_wrf_basin_season_dict[var] = era5_wrf_basin_season_dict[var].sel(day = slice('2001','2023'))

    # era5_wrf_basin_season_dict[var] = era5_wrf_basin_season_dict[var].sel(day = [pd.to_datetime(i).month in list(range(6,10)) for i in era5_wrf_basin_season_dict[var].day.values])
era5_wrf_basin_season_dict[var]


In [ ]:
var_list = ['e','e_s','rh','vpd','wspd','wspdmax','soil_m','SOIL_M','hdw','fwi','ffwi','mffwi','haines','kbdi',"fm1","fm10","fm100","fm1000","erc","bi","sc","ic","ros",'isi','bui','dc','ffmc','dmc']

era5_wrf_basin_pct_rank_dict = {}

for var in var_list:
#     era5_wrf_basin_dict[var] = xr.open_dataset(f'daily_{var}_by_pyrome_with_ignitions.nc').load()
    era5_wrf_basin_pct_rank_dict[var] = xr.open_dataset(f'data/daily_{var}_by_pyrome_with_ignitions.nc').load()
    # era5_wrf_basin_pct_rank_dict[var] = era5_wrf_basin_pct_rank_dict[var].sel(day = [pd.to_datetime(i).month in list(range(6,11)) for i in era5_wrf_basin_pct_rank_dict[var].day.values])
    
    for pyrome_id in sorted(pyrome['PYROME']):
        era5_wrf_basin_pct_rank_dict[var].sel(basin = pyrome_id)['fwi'][:] = era5_wrf_basin_pct_rank_dict[var].sel(basin = pyrome_id)['fwi'].rank(dim = 'day', pct = True)
    
    if var in ['e','soil_m','rh','fm1','fm10','fm100','fm1000']:
        era5_wrf_basin_pct_rank_dict[var]['fwi'] = 1-(era5_wrf_basin_pct_rank_dict[var]['fwi'])

#     era5_wrf_basin_dict[var] = era5_wrf_basin_dict[var].sel(day = slice('1980','2020'))
era5_wrf_basin_pct_rank_dict['rh']

In [ ]:
era5_wrf_basin_anomaly_pct_rank_dict = {}

var_list = ['e','e_s','rh','vpd','wspd','wspdmax','soil_m','SOIL_M','hdw','fwi','ffwi','mffwi','haines','kbdi',"fm1","fm10","fm100","fm1000","erc","bi","sc","ic","ros",'isi','bui','dc','ffmc','dmc']

for var in var_list:
    era5_wrf_basin_anomaly_pct_rank_dict[var] = xr.open_dataset(f'data/daily_{var}_by_pyrome_with_ignitions.nc')
    era5_wrf_basin_anomaly_pct_rank_dict[var] = era5_wrf_basin_anomaly_pct_rank_dict[var].load()
    era5_wrf_basin_anomaly_pct_rank_dict[var] = era5_wrf_basin_anomaly_pct_rank_dict[var].groupby('day.dayofyear') - era5_wrf_basin_anomaly_pct_rank_dict[var].groupby('day.dayofyear').mean()
    era5_wrf_basin_anomaly_pct_rank_dict[var] = era5_wrf_basin_anomaly_pct_rank_dict[var].sel(day = slice('2001','2023'))
    
    for pyrome_id in sorted(pyrome['PYROME']):
        era5_wrf_basin_anomaly_pct_rank_dict[var].sel(basin = pyrome_id)['fwi'][:] = era5_wrf_basin_anomaly_pct_rank_dict[var].sel(basin = pyrome_id)['fwi'].rank(dim = 'day', pct = True)
        
    if var in ['e','soil_m','rh','fm1','fm10','fm100','fm1000']:
        era5_wrf_basin_anomaly_pct_rank_dict[var]['fwi'] = 1-(era5_wrf_basin_anomaly_pct_rank_dict[var]['fwi'])

    # era5_wrf_basin_anomaly_pct_rank_dict[var] = era5_wrf_basin_anomaly_pct_rank_dict[var].sel(day = [pd.to_datetime(i).month in list(range(6,10)) for i in era5_wrf_basin_anomaly_pct_rank_dict[var].day.values])
era5_wrf_basin_anomaly_pct_rank_dict[var]


In [ ]:
era5_wrf_basin_season_pct_rank_dict = {}

var_list = ['e','e_s','rh','vpd','wspd','wspdmax','soil_m','SOIL_M','hdw','fwi','ffwi','mffwi','haines','kbdi',"fm1","fm10","fm100","fm1000","erc","bi","sc","ic","ros",'isi','bui','dc','ffmc','dmc']

for var in var_list:
    era5_wrf_basin_season_pct_rank_dict[var] = xr.open_dataset(f'data/daily_{var}_by_pyrome_with_ignitions.nc')
    era5_wrf_basin_season_pct_rank_dict[var] = era5_wrf_basin_season_pct_rank_dict[var].load()
    era5_wrf_basin_season_pct_rank_dict[var] = era5_wrf_basin_season_pct_rank_dict[var] - (era5_wrf_basin_season_pct_rank_dict[var].groupby('day.dayofyear') - era5_wrf_basin_season_pct_rank_dict[var].groupby('day.dayofyear').mean())
    era5_wrf_basin_season_pct_rank_dict[var] = era5_wrf_basin_season_pct_rank_dict[var].sel(day = slice('2001','2023'))
    
    for pyrome_id in sorted(pyrome['PYROME']):
        era5_wrf_basin_season_pct_rank_dict[var].sel(basin = pyrome_id)['fwi'][:] = era5_wrf_basin_season_pct_rank_dict[var].sel(basin = pyrome_id)['fwi'].rank(dim = 'day', pct = True)
        
    if var in ['e','soil_m','rh','fm1','fm10','fm100','fm1000']:
        era5_wrf_basin_season_pct_rank_dict[var]['fwi'] = 1-(era5_wrf_basin_season_pct_rank_dict[var]['fwi'])

    # era5_wrf_basin_season_pct_rank_dict[var] = era5_wrf_basin_season_pct_rank_dict[var].sel(day = [pd.to_datetime(i).month in list(range(6,10)) for i in era5_wrf_basin_season_pct_rank_dict[var].day.values])
era5_wrf_basin_season_pct_rank_dict[var]


In [ ]:

FIRED_forest = FIRED[['Forest' in FIRED['lc_name'].iloc[i] for i in range(len(FIRED))]]
# FIRED_forest = FIRED[[('Forest' in FIRED['lc_name'].iloc[i]) or ('Woody Savannas' in FIRED['lc_name'].iloc[i]) for i in range(len(FIRED))]]
FIRED_non_forest = FIRED[[('Forest' not in FIRED['lc_name'].iloc[i]) & ('Crop' not in FIRED['lc_name'].iloc[i]) for i in range(len(FIRED))]]
FIRED_shrub = FIRED[[('Shrub' in FIRED['lc_name'].iloc[i]) for i in range(len(FIRED))]]
FIRED_grass = FIRED[[('Grass' in FIRED['lc_name'].iloc[i]) for i in range(len(FIRED))]]
FIRED_woody_savanna = FIRED[[('Woody Savannas' in FIRED['lc_name'].iloc[i]) for i in range(len(FIRED))]]
FIRED_savanna = FIRED[[('Savanna' in FIRED['lc_name'].iloc[i]) for i in range(len(FIRED))]]
FIRED_crop = FIRED[[('Crop' in FIRED['lc_name'].iloc[i]) for i in range(len(FIRED))]]
# FIRED_non_forest = FIRED[[('Forest' not in FIRED['lc_name'].iloc[i]) & ('Urban' not in FIRED['lc_name'].iloc[i]) for i in range(len(FIRED))]]
# FIRED_forest = FIRED[['Forest' in data['lc_name'].iloc[i] for i in range(len(data))]]

np.unique(FIRED['lc_name'])
# np.sum(['Urban' in FIRED['lc_name'].iloc[i] for i in range(len(FIRED))])
# np.sum([('Forest' not in FIRED['lc_name'].iloc[i]) & ('Urban' not in FIRED['lc_name'].iloc[i]) for i in range(len(FIRED))])



In [ ]:
len(np.unique(FIRED['id'])), len(FIRED)

In [ ]:
pyrome_by_regions = {}
pyrome_by_regions['all'] = sorted(pyrome['PYROME'])
pyrome_by_regions['pnw'] = [1,2,3,4,5,6,16,19]
pyrome_by_regions['rockies'] = [7,8,9,10,11,12,13,14,15,20,59,60,65,128]
pyrome_by_regions['ca_nv'] = [17,18,21,22,23,25,26,27,28,29,30,31,32,33,34,35,36,41,77,78]
pyrome_by_regions['four_corners'] = [24,37,38,39,40,42,43,44,45,48,127]


In [ ]:
annual_risk_list = []
for pyrome_id in sorted(pyrome['PYROME']):
    data = FIRED_forest[(FIRED_forest['PYROME'] == pyrome_id)][['event_day','id','year']].reset_index()
    annual_risk_list += [len(np.unique(data[(data['event_day'] == 1) & (data['year'] < 2023)]['id']))]

pyrome['forest_fire_count'] = annual_risk_list
pyrome['forest_fire_count_per_year'] = np.array(annual_risk_list)/22
pyrome['forest_fire_pct'] = pyrome['forest_fire_count'] / pyrome['fire_count']

annual_risk_list = []
for pyrome_id in sorted(pyrome['PYROME']):
    data = FIRED_non_forest[(FIRED_non_forest['PYROME'] == pyrome_id)][['event_day','id','year']].reset_index()
    annual_risk_list += [len(np.unique(data[(data['event_day'] == 1) & (data['year'] < 2023)]['id']))]

pyrome['non_forest_fire_count'] = annual_risk_list
pyrome['non_forest_fire_count_per_year'] = np.array(annual_risk_list)/22

pyrome_region_list = []
for pyrome_id in sorted(pyrome['PYROME']):
    i = 0
    for region in ['pnw','rockies','ca_nv','four_corners']:
        if pyrome_id in pyrome_by_regions[region]:
            pyrome_region_list += [i]
            break
        i += 1
pyrome['region'] = pyrome_region_list
pyrome.head()

In [ ]:
# land_mask = xr.where(era5_wrf.SOIL_M.max(dim = 'day') > 0, 1, 0)
# land_mask = land_mask.load()

fig,ax=plt.subplots(1,1,sharex=True,sharey=True,figsize=(4,6), dpi = 250, subplot_kw={'projection': ccrs.PlateCarree()})
    
bounds = list(range(0,51,5))
cmap = cm.get_cmap('Reds', 10)
colors = cmap(np.arange(0,cmap.N))
norm = matplotlib.colors.BoundaryNorm(boundaries=bounds, ncolors=10)

p = pyrome.plot(ax = ax, column = 'fire_count_per_year', cmap = cmap, norm = norm, edgecolor = 'k', linewidth = 1)
# p = pyrome.plot(ax = ax, column = 'region', cmap = cmap, vmin = 0, vmax = 4, edgecolor = 'k', linewidth = 1)
# p = pyrome.plot(ax = ax, column = 'forest_fire_count_per_year', cmap = cmap, vmin = 0, vmax = 10, edgecolor = 'k', linewidth = 1)

cax = fig.add_axes([0.93, 0.18, 0.05, 0.63])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm._A = []
cbar = fig.colorbar(sm, cax=cax, extend = 'max')
cbar.set_label('Number of fires per year (2001-2022)', rotation = 270, labelpad = 15)

ax.set_title('Pyromes in western US', fontsize = 15)

In [ ]:
fig,ax = plt.subplots(5,6,figsize = (15,10.5), dpi = 400)

colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive', 'lightgray', 'darkcyan', 'darkred', 'green']

# var_list = ['e','e_s','rh','vpd','uv10','soil_m','hdw','haines','FFWI','kbdi','mffwi','fwi']
# var_list = ['e','e_s','rh','vpd','uv10','soil_m','hdw','fwi','FFWI','mffwi','haines','kbdi',"fm1","fm10","fm100","fm1000"]
# var_list = ['e','e_s','rh','vpd','uv10','soil_m','hdw','fwi','ffwi','mffwi','haines','kbdi',"fm1","fm10","fm100","fm1000","erc","bi","sc","ic","ros"]
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','ffmc','dmc','dc','isi','bui','fwi',"fm100","fm1000","erc","bi"]
# var_list = ['e_s','rh','vpd','SOIL_M','soil_m','hdw','haines','kbdi','ffwi','mffwi','fwi',"fm100","fm1000","erc","bi"]

count_pdf_dict = {}
count_cdf_dict = {}

auc_df = pd.DataFrame()
auc_list = []
mean_auc_list = []
auc_std_list = []
tick_color_list = []

i = 0
for var in var_list:
#     if i < 6:
#         row, col = i // 2, i % 2
#     else:
#         row, col = (i-6) // 2, 2 + (i-6) % 2
    row,col = i//6, i%6
    mean = np.zeros(100)
    
    count_cdf_dict[var] = pd.DataFrame()
    count_pdf_dict[var] = pd.DataFrame()
    
    var_auc_list = []
    
    mean_entropy = 0
    for pyrome_id in sorted(pyrome['PYROME']):
        data = FIRED[(FIRED['PYROME'] == pyrome_id) & (FIRED['event_day'] == 1)][['ig_date','event_day','id','month']]
        # data = data[data['ig_date'] <= '2021-08-31']

        # data = data[(data['month'] >= 6) & (data['month'] <= 10)]
        count_pdf_dict[var][pyrome_id] = np.histogram(1 - era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = data['ig_date'].values).values, bins = 100, range = (0,1), density = True)[0]
        count_cdf_dict[var][pyrome_id] = np.cumsum(np.histogram(1 - era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = data['ig_date'].values).values, bins = 100, range = (0,1), density = True)[0])
        ax[row,col].plot(count_cdf_dict[var][pyrome_id], c = 'royalblue', linewidth = 0.4, alpha = 0.9)
        
        mean_entropy += entropy(count_pdf_dict[var][pyrome_id])/np.log(len(count_pdf_dict[var][pyrome_id]))/len(pyrome)
        
        # if var == 'soil_m':
        #     auc_list += [np.sum(count_cdf_dict[var][pyrome_id] - np.linspace(1,100,100))/50/100]
        var_auc_list += [np.sum(count_cdf_dict[var][pyrome_id] - np.linspace(1,100,100))/50/100]
        
    ax[row,col].plot(count_cdf_dict[var].mean(axis = 1), c = 'k', 
                     label = f'{round(np.sum(count_cdf_dict[var].mean(axis = 1) - np.linspace(1,100,100))/50/100,2)}')
#                      label = round(mean_entropy,3))
#                      label = round(entropy(count_pdf_dict[var].median(axis = 1))/np.log(len(count_pdf_dict[var].median(axis = 1))),3))
#     ax[row,col].plot(mean, c = 'k', 
#                      label = f'{round(np.sum((mean) - np.linspace(1,100,100))/50/100,2)}')
    ax[row,col].set_title(var.upper(), fontsize = 15, y = .98)
    auc_df[var] = var_auc_list
    
    mean_auc_list += [np.sum(count_cdf_dict[var].mean(axis = 1) - np.linspace(1,100,100))/50/100]
    auc_std_list += [np.std(var_auc_list)]
    
    for j in np.arange(0,101,10):
        ax[row,col].axhline(j, c = 'grey', linewidth = 0.1, linestyle = '--')
    for j in np.arange(0,101,10):
        ax[row,col].axvline(j, c = 'grey', linewidth = 0.1, linestyle = '--')
    ax[row,col].fill_between(range(0,100),range(0,100),100*np.ones(100), color = 'cornflowerblue', alpha = 0.15, linestyle = '--')
    ax[row,col].fill_between(range(0,100),np.zeros(100),range(0,100), color = 'sandybrown', alpha = 0.4, linestyle = '--')

    if row == 0:
        ax[row,col].set_position(ax[row,col].get_position(original = True).bounds + np.array([0, 0.03, 0, 0.00]))
        
    ax[row,col].set_xticks([0,50,100])
    ax[row,col].set_yticks([0,50,100])
    if col == 0:
        ax[row,col].set_yticklabels([0,50,100])
    else:
        ax[row,col].set_yticklabels([])
        
    if row == 3:
        ax[row,col].set_xticklabels([100,50,0])
    elif (row == 2) & (col >= 3):
        ax[row,col].set_xticklabels([100,50,0])
    else:
        ax[row,col].set_xticklabels([])
    ax[row,col].legend(loc = 'lower right', fontsize = 13, handlelength = 1)
    
    if col == 0:
        tick_color_list += ['blue']
    else:
        tick_color_list += ['black']
    i += 1

auc_df.index = sorted(pyrome['PYROME'])

# ax[1,0].set_ylabel('Percent of Ignitions Captured', fontsize = 15)
ax[3,2].text(1.1,-0.25,'Index Percentile', fontsize = 13,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[3,2].transAxes)
ax[1,0].text(-0.34,0.0,'Percent of Ignitions Captured', fontsize = 13,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[1,0].transAxes, rotation = 90)

ax[0,0].text(-0.55,0.5,'Hydrometeorological\nVariables', fontsize = 20,
            horizontalalignment='right',
            verticalalignment='center', transform=ax[0,0].transAxes)
ax[2,0].text(-0.55,.5,'Fire Indices', fontsize = 20,
            horizontalalignment='right',
            verticalalignment='center', transform=ax[2,0].transAxes)

auc_sorted = [i[0] for i in sorted(zip(mean_auc_list,colors,var_list,auc_std_list),key=lambda x:x[0])]
color_sorted = [i[1] for i in sorted(zip(mean_auc_list,colors,var_list,auc_std_list),key=lambda x:x[0])]
var_sorted = [i[2].upper() for i in sorted(zip(mean_auc_list,colors,var_list,auc_std_list),key=lambda x:x[0])]    
auc_std_list_sorted = [i[3] for i in sorted(zip(mean_auc_list,colors,var_list,auc_std_list),key=lambda x:x[0])]

for threshold in np.arange(-0.1,0.46,0.05):
    ax[4,0].axvline(threshold, linestyle = '--', linewidth = 0.5, c= 'grey')
ax[4,0].axvline(0, linestyle = '-', linewidth = 1.5, c = 'black')

p = ax[4,0].barh(range(len(mean_auc_list)), np.array(auc_sorted), left = 0, color = color_sorted, alpha = 0.8)
# ax[4,0].bar_label(p, labels=[round(i,2) for i in auc_sorted][1:], label_type='edge',fontsize = 9, padding = 3)
# ax[4,0].bar_label(p, labels=[f'±{round(i,2)}' for i in auc_std_list_sorted][1:], label_type='center',fontsize = 8)
ax[4,0].bar_label(p, labels=['',''] + [f'{round(auc_sorted[i],2)}±{round(auc_std_list_sorted[i],2)}' for i in range(2,len(auc_std_list_sorted))], label_type='edge',fontsize = 10, padding = 5)
# ax[4,0].errorbar(range(len(mean_auc_list)),auc_sorted, yerr = auc_std_list_sorted, linewidth = 0, color = 'k', elinewidth = 2)

ax[4,0].text(0.006,0,f'{round(auc_sorted[0],2)}±{round(auc_std_list_sorted[0],2)}', fontsize = 10,
            horizontalalignment='left',
            verticalalignment='center')
ax[4,0].text(0.006,1,f'{round(auc_sorted[1],2)}±{round(auc_std_list_sorted[1],2)}', fontsize = 10,
            horizontalalignment='left',
            verticalalignment='center')

ax[4,0].set_yticks(range(len(mean_auc_list)))
ax[4,0].set_yticklabels(var_sorted,rotation = 0,fontsize = 12)

ax[4,0].yaxis.tick_right()
ax[4,0].yaxis.set_ticks_position('both')
# ax[4,0].tick_params(labeltop=False, labelright=True)

ax[4,0].set_xticks(np.arange(-0.1,0.41,0.1))
ax[4,0].set_xticklabels([round(i,1) for i in np.arange(-0.1,0.41,0.1)],fontsize = 12)
ax[4,0].set_xlabel('AUC score',fontsize = 13)
    
ax[4,0].set_xlim([-0.1,0.5])

for ticklabel in ax[4,0].get_yticklabels():
    if ticklabel.get_text() in [i.upper() for i in ['e_s','rh','vpd','wspd','wspdmax','soil_m']]:
        ticklabel.set_color('blue')
            
ax[4,0].set_position([0.1,-0.18,0.7,0.38])
# ax[4,0].set_position([0.8,0.1,0.3,0.8])

for col in range(3,6):
    plt.delaxes(ax[3,col])
for col in range(1,6):
    plt.delaxes(ax[4,col])


# plt.delaxes(ax[1,5])
# plt.delaxes(ax[2,5])
ax[0,0].text(-0.4,1.2,'a', fontsize = 28,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[0,0].transAxes)
ax[4,0].text(-0.04,1.05,'b', fontsize = 28,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[4,0].transAxes)


In [ ]:
# auc_df = pd.DataFrame()
auc_dict = {}

var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]

for vegetation in ['all','forest','non_forest']:
    print(vegetation)
    auc_dict[vegetation] = {}
    if vegetation == 'all':
        df_tmp = FIRED
    else:
        df_tmp = eval(f'FIRED_{vegetation}')
    
    for season_type in ['total','season','anomaly']:
        i = 0
        auc_dict[vegetation][season_type] = pd.DataFrame()
        for var in var_list:
            var_auc_list = []
            data = pd.DataFrame()
            for pyrome_id in sorted(pyrome['PYROME']):
                tmp = df_tmp[(df_tmp['PYROME'] == pyrome_id) & (df_tmp['event_day'] == 1)][['ig_date','event_day','id']]
                if season_type == 'total':
                    tmp['fwi'] = era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = tmp['ig_date'].values).values
                else:
                    tmp['fwi'] = eval(f'era5_wrf_basin_{season_type}_pct_rank_dict')[var]['fwi'].sel(basin = pyrome_id).sel(day = tmp['ig_date'].values).values

                # tmp['fwi'] = era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = tmp['ig_date'].values).values
                data = pd.concat([data,tmp])

            # for pyrome_id in sorted(pyrome['PYROME']):
            #     if vegetation == 'all': 
            #         data = FIRED[(FIRED['PYROME'] == pyrome_id) & (FIRED['event_day'] == 1)][['ig_date','event_day','id']]
            #     else:
            #         data = eval(f'FIRED_{vegetation}')[(eval(f'FIRED_{vegetation}')['PYROME'] == pyrome_id) & (eval(f'FIRED_{vegetation}')['event_day'] == 1)][['ig_date','event_day','id']]
                
            #     if season_type == 'total':
            #         tmp = np.cumsum(np.histogram(1 - era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = data['ig_date'].values).values, bins = 100, range = (0,1), density = True)[0])
            #     else:
            #         tmp = np.cumsum(np.histogram(1 - eval(f'era5_wrf_basin_{season_type}_pct_rank_dict')[var]['fwi'].sel(basin = pyrome_id).sel(day = data['ig_date'].values).values, bins = 100, range = (0,1), density = True)[0])

            tmp = np.cumsum(np.histogram(1 - data['fwi'], bins = 100, range = (0,1), density = True)[0])
            var_auc = np.sum(tmp - np.linspace(1,100,100))/50/100
            auc_dict[vegetation][season_type][var] = [var_auc]

            i += 1

        # auc_dict[vegetation][season_type].index = sorted(pyrome['PYROME'])
print('Done')

In [ ]:

colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive','teal', 'darkcyan', 'darkred', 'green', 'lightgray']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]

count_pdf_anomaly_dict = {}
count_cdf_anomaly_dict = {}

auc_df = pd.DataFrame()
auc_list = []
mean_auc_list = []
auc_std_list = []
tick_color_list = []

i = 0
for var in var_list:

    row,col = i//6, i%6
    mean = np.zeros(100)
    
    count_cdf_anomaly_dict[var] = pd.DataFrame()
    count_pdf_anomaly_dict[var] = pd.DataFrame()
    
    var_auc_list = []
    
    mean_entropy = 0
    for pyrome_id in sorted(pyrome['PYROME']):
        data = FIRED[(FIRED['PYROME'] == pyrome_id) & (FIRED['event_day'] == 1)][['ig_date','event_day','id']]
        count_pdf_anomaly_dict[var][pyrome_id] = np.histogram(1 - era5_wrf_basin_anomaly_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = data['ig_date'].values).values, bins = 100, range = (0,1), density = True)[0]
        count_cdf_anomaly_dict[var][pyrome_id] = np.cumsum(np.histogram(1 - era5_wrf_basin_anomaly_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = data['ig_date'].values).values, bins = 100, range = (0,1), density = True)[0])
        ax[row,col].plot(count_cdf_anomaly_dict[var][pyrome_id], c = 'royalblue', linewidth = 0.4, alpha = 0.9)
        
        mean_entropy += entropy(count_pdf_anomaly_dict[var][pyrome_id])/np.log(len(count_pdf_anomaly_dict[var][pyrome_id]))/len(pyrome)
        
        # if var == 'soil_m':
        #     auc_list += [np.sum(count_cdf_anomaly_dict[var][pyrome_id] - np.linspace(1,100,100))/50/100]
        var_auc_list += [np.sum(count_cdf_anomaly_dict[var][pyrome_id] - np.linspace(1,100,100))/50/100]
    auc_df[var] = var_auc_list
    
    mean_auc_list += [np.sum(count_cdf_anomaly_dict[var].mean(axis = 1) - np.linspace(1,100,100))/50/100]
    auc_std_list += [np.std(var_auc_list)]
    
    i += 1

auc_df.index = sorted(pyrome['PYROME'])


In [ ]:
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]

count_pdf_season_dict = {}
count_cdf_season_dict = {}

auc_df = pd.DataFrame()
auc_list = []
mean_auc_list = []
auc_std_list = []
tick_color_list = []

i = 0
for var in var_list:
#     if i < 6:
#         row, col = i // 2, i % 2
#     else:
#         row, col = (i-6) // 2, 2 + (i-6) % 2
    row,col = i//6, i%6
    mean = np.zeros(100)
    
    count_cdf_season_dict[var] = pd.DataFrame()
    count_pdf_season_dict[var] = pd.DataFrame()
    
    var_auc_list = []
    
    mean_entropy = 0
    for pyrome_id in sorted(pyrome['PYROME']):
        data = FIRED[(FIRED['PYROME'] == pyrome_id) & (FIRED['event_day'] == 1)][['ig_date','event_day','id']]
        count_pdf_season_dict[var][pyrome_id] = np.histogram(1 - era5_wrf_basin_season_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = data['ig_date'].values).values, bins = 100, range = (0,1), density = True)[0]
        count_cdf_season_dict[var][pyrome_id] = np.cumsum(np.histogram(1 - era5_wrf_basin_season_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = data['ig_date'].values).values, bins = 100, range = (0,1), density = True)[0])
        ax[row,col].plot(count_cdf_season_dict[var][pyrome_id], c = 'royalblue', linewidth = 0.4, alpha = 0.9)
        
        mean_entropy += entropy(count_pdf_season_dict[var][pyrome_id])/np.log(len(count_pdf_season_dict[var][pyrome_id]))/len(pyrome)
        
        # if var == 'soil_m':
        #     auc_list += [np.sum(count_cdf_season_dict[var][pyrome_id] - np.linspace(1,100,100))/50/100]
        var_auc_list += [np.sum(count_cdf_season_dict[var][pyrome_id] - np.linspace(1,100,100))/50/100]
        
    auc_df[var] = var_auc_list
    
    mean_auc_list += [np.sum(count_cdf_season_dict[var].mean(axis = 1) - np.linspace(1,100,100))/50/100]
    auc_std_list += [np.std(var_auc_list)]
    
    i += 1

auc_df.index = sorted(pyrome['PYROME'])


In [ ]:
# fig,ax = plt.subplots(3,1,figsize = (8,9), dpi = 400, sharex = True)
fig,ax = plt.subplots(1,3,figsize = (14,4.5), dpi = 400, sharex = True)

# var_list = ['e_s','rh','vpd','wspd','soil_m','hdw','haines','kbdi','ffwi','mffwi','fwi',"fm100","fm1000","erc","bi"]
# colors = ['violet','orangered','royalblue','orange','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive']
colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive', 'lightgray', 'darkcyan', 'darkred', 'green']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]

mean_auc_list = []
for var in var_list:
    var_auc_list = []
    for pyrome_id in sorted(pyrome['PYROME']):
        # tmp = np.cumsum(np.histogram(1 - count_cdf_dict[var][pyrome_id], bins = 100, range = (0,1), density = True)[0])
        var_auc = np.sum(count_cdf_dict[var][pyrome_id] - np.linspace(1,100,100))/50/100
        var_auc_list += [var_auc]
        
    mean_auc_list += [np.mean(var_auc_list)]
# mean_auc_list = [auc_dict['all']['total'][var].values[0] for var in var_list]

auc_sorted = [i[0] for i in sorted(zip(mean_auc_list,colors,var_list),key=lambda x:x[0])][::-1]
color_sorted = [i[1] for i in sorted(zip(mean_auc_list,colors,var_list),key=lambda x:x[0])][::-1]
var_sorted = [i[2].upper() for i in sorted(zip(mean_auc_list,colors,var_list),key=lambda x:x[0])][::-1]

i = 0
for var in var_sorted:
    var = var.lower()
    var_auc_list = []
    for pyrome_id in sorted(pyrome['PYROME']):
        # tmp = np.cumsum(np.histogram(1 - count_cdf_dict[var][pyrome_id], bins = 100, range = (0,1), density = True)[0])
        var_auc = np.sum(count_cdf_dict[var][pyrome_id] - np.linspace(1,100,100))/50/100
        var_auc_list += [var_auc]
    p = ax[0].barh(len(var_list) - i, np.mean(var_auc_list), color = color_sorted[i], height = 0.75)
    if np.mean(var_auc_list) > 0:
        ax[0].bar_label(p, labels=[f'{round(np.mean(var_auc_list),2)}'], label_type='edge',fontsize = 13, padding = 5)
    else:
        ax[0].text(0.01,len(var_list) - i,f'{round(np.mean(var_auc_list),2)}', fontsize = 13,
            horizontalalignment='left',
            verticalalignment='center')
        
    var_auc_list = []
    for pyrome_id in sorted(pyrome['PYROME']):
        var_auc = np.sum(count_cdf_season_dict[var][pyrome_id] - np.linspace(1,100,100))/50/100
        var_auc_list += [var_auc]
    p = ax[1].barh(len(var_list) - i, np.mean(var_auc_list), color = color_sorted[i], height = 0.75)
    if np.mean(var_auc_list) > 0:
        ax[1].bar_label(p, labels=[f'{round(np.mean(var_auc_list),2)}'], label_type='edge',fontsize = 13, padding = 5)
    else:
        ax[1].text(0.01,len(var_list) - i,f'{round(np.mean(var_auc_list),2)}', fontsize = 13,
            horizontalalignment='left',
            verticalalignment='center')
        
    var_auc_list = []
    for pyrome_id in sorted(pyrome['PYROME']):
        var_auc = np.sum(count_cdf_anomaly_dict[var][pyrome_id] - np.linspace(1,100,100))/50/100
        var_auc_list += [var_auc]
    p = ax[2].barh(len(var_list) - i, np.mean(var_auc_list), color = color_sorted[i], height = 0.75)
    ax[2].bar_label(p, labels=[f'{round(np.mean(var_auc_list),2)}'], label_type='edge',fontsize = 13, padding = 5)
    
    i += 1

for i in range(3):   
    ax[i].set_yticks(range(1,len(var_list)+1))
    ax[i].set_xlim([-0.25,0.65])
    ax[i].set_yticklabels([i.upper() for i in var_sorted[::-1]], fontsize = 14)

    if i < 2:
        ax[i].tick_params(axis='y',which='both',direction = 'out',right=True, left=True)
ax[1].set_yticklabels([])
ax[2].yaxis.tick_right()
ax2 = ax[2].secondary_yaxis("left")
ax2.set_yticks(range(1,len(var_list)+1))
ax2.set_yticklabels([])
ax2.tick_params(axis="y", direction="out")

# ax[2].set_yticklabels([i.upper() for i in var_list][::-1])

ax[2].yaxis.tick_right()

for i in range(3):
    for ticklabel in ax[i].get_yticklabels():
        if ticklabel.get_text() in [j.upper() for j in ['e_s','rh','vpd','wspd','wspdmax','soil_m']]:
            ticklabel.set_color('blue')

    ax[i].set_xticks(np.arange(-0.2,0.61,0.1))
    ax[i].set_xticklabels([round(i,1) for i in np.arange(-0.2,0.61,0.1)],fontsize = 12)
    for threshold in np.arange(-0.2,0.61,0.1):
        ax[i].axvline(threshold, linestyle = '--', linewidth = 0.5, c= 'grey', zorder = 0)
    for threshold in np.arange(-0.15,0.61,0.1):
        ax[i].axvline(threshold, linestyle = '--', linewidth = 0.2, c= 'grey', zorder = 0)
    ax[i].axvline(0, linestyle = '-', linewidth = 1.5, c = 'black')
    ax[i].set_position([i*0.33,0,0.31,1])

ax[0].text(0.05,1.08,'a', fontsize = 26,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[0].transAxes)
ax[1].text(0.05,1.08,'b', fontsize = 26,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[1].transAxes)
ax[2].text(0.05,1.08,'c', fontsize = 26,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[2].transAxes)

ax[0].set_title('Daily Values', fontsize = 25, y = 1.03)
ax[1].set_title('Seasonality', fontsize = 25, y = 1.03)
ax[2].set_title('Anomalies', fontsize = 25, y = 1.03)

ax[1].set_xlabel('AUC score', fontsize = 20, labelpad = 10) 
# ax[2].set_xlabel('AUC score', fontsize = 18, labelpad = 10) 


In [ ]:
# fig,ax = plt.subplots(3,1,figsize = (8,9), dpi = 400, sharex = True)
fig,ax = plt.subplots(3,3,figsize = (16,16), dpi = 400, sharex = True)

colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive', 'lightgray', 'darkcyan', 'darkred', 'green']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]

row = 0
for vegetation in ['all','forest','non_forest']:
    mean_auc_list = [np.mean(auc_dict[vegetation]['total'][var]) for var in var_list]

    auc_sorted = [i[0] for i in sorted(zip(mean_auc_list,colors,var_list),key=lambda x:x[0])][::-1]
    color_sorted = [i[1] for i in sorted(zip(mean_auc_list,colors,var_list),key=lambda x:x[0])][::-1]
    var_sorted = [i[2].upper() for i in sorted(zip(mean_auc_list,colors,var_list),key=lambda x:x[0])][::-1]

    i = 0
    for var in var_sorted:

        var = var.lower()
        p = ax[row,0].barh(len(var_list) - i, np.mean(auc_dict[vegetation]['total'][var]), color = color_sorted[i], height = 0.75)
        if np.mean(auc_dict[vegetation]['total'][var]) > 0:
            ax[row,0].bar_label(p, labels=[f'{round(np.mean(auc_dict[vegetation]['total'][var]),2)}'], label_type='edge',fontsize = 13, padding = 5)
        else:
            ax[row,0].text(0.01,len(var_list) - i,f'{round(np.mean(auc_dict[vegetation]['total'][var]),2)}', fontsize = 13,
                horizontalalignment='left',
                verticalalignment='center')
            
        p = ax[row,1].barh(len(var_list) - i, np.mean(auc_dict[vegetation]['season'][var]), color = color_sorted[i], height = 0.75)
        if np.mean(auc_dict[vegetation]['season'][var]) > 0:
            ax[row,1].bar_label(p, labels=[f'{round(np.mean(auc_dict[vegetation]['season'][var]),2)}'], label_type='edge',fontsize = 13, padding = 5)
        else:
            ax[row,1].text(0.01,len(var_list) - i,f'{round(np.mean(auc_dict[vegetation]['season'][var]),2)}', fontsize = 13,
                horizontalalignment='left',
                verticalalignment='center')
        
        p = ax[row,2].barh(len(var_list) - i, np.mean(auc_dict[vegetation]['anomaly'][var]), color = color_sorted[i], height = 0.75)
        if np.mean(auc_dict[vegetation]['anomaly'][var]) > 0:
            ax[row,2].bar_label(p, labels=[f'{round(np.mean(auc_dict[vegetation]['anomaly'][var]),2)}'], label_type='edge',fontsize = 13, padding = 5)
        else:
            ax[row,2].text(0.01,len(var_list) - i,f'{round(np.mean(auc_dict[vegetation]['anomaly'][var]),2)}', fontsize = 13,
                horizontalalignment='left',
                verticalalignment='center')
        i += 1

    for i in range(3):   
        ax[row,i].set_yticks(range(1,len(var_sorted)+1))
        ax[row,i].set_xlim([-0.25,0.72])
        ax[row,i].set_yticklabels([i.upper() for i in var_sorted][::-1], fontsize = 14)

        if i < 2:
            ax[row,i].tick_params(axis='y',which='both',direction = 'out',right=True, left=True)
    ax[row,1].set_yticklabels([])
    ax[row,2].yaxis.tick_right()
    ax2 = ax[row,2].secondary_yaxis("left")
    ax2.set_yticks(range(1,len(var_list)+1))
    ax2.set_yticklabels([])
    ax2.tick_params(axis="y", direction="out")

    # ax[2].set_yticklabels([i.upper() for i in var_list][::-1])

    ax[row,2].yaxis.tick_right()

    for i in range(3):
        for ticklabel in ax[row,i].get_yticklabels():
            if ticklabel.get_text() in [j.upper() for j in ['e_s','rh','vpd','wspd','wspdmax','soil_m']]:
                ticklabel.set_color('blue')

        ax[row,i].set_xticks(np.arange(-0.2,0.71,0.1))
        ax[row,i].set_xticklabels([round(i,1) for i in np.arange(-0.2,0.71,0.1)],fontsize = 12)
        for threshold in np.arange(-0.2,0.71,0.1):
            ax[row,i].axvline(threshold, linestyle = '--', linewidth = 0.5, c= 'grey', zorder = 0)
        for threshold in np.arange(-0.15,0.71,0.1):
            ax[row,i].axvline(threshold, linestyle = '--', linewidth = 0.2, c= 'grey', zorder = 0)
        ax[row,i].axvline(0, linestyle = '-', linewidth = 1.5, c = 'black')
        ax[row,i].set_position([i*0.33,1-(row+1)*0.33,0.31,0.26])


    row += 1

letter_list = ['a','b','c','d','e','f','g','h','i','j','k']
for i in range(9):
    row,col = i//3,i%3
    
    ax[row,col].text(0.05,1.08,letter_list[i], fontsize = 26,
                horizontalalignment='center',
                verticalalignment='center', transform=ax[row,col].transAxes)

ax[0,0].set_title('Daily Values', fontsize = 25, y = 1.04)
ax[0,1].set_title('Seasonality', fontsize = 25, y = 1.04)
ax[0,2].set_title('Anomalies', fontsize = 25, y = 1.04)

ax[0,0].text(-0.3,0.5,'All', fontsize = 24,
                horizontalalignment='right',
                verticalalignment='center', transform=ax[0,0].transAxes)
ax[1,0].text(-0.3,0.5,'Forest', fontsize = 24,
                horizontalalignment='right',
                verticalalignment='center', transform=ax[1,0].transAxes)
ax[2,0].text(-0.3,0.5,'Non-Forest', fontsize = 24,
                horizontalalignment='right',
                verticalalignment='center', transform=ax[2,0].transAxes)

ax[2,1].set_xlabel('AUC score', fontsize = 18, labelpad = 10)


In [ ]:
fig,ax = plt.subplots(2,3,figsize = (16,11), dpi = 400, sharex = False, sharey = False)

# var_list = ['e','e_s','rh','vpd','uv10','soil_m','hdw','haines','FFWI','kbdi','mffwi','fwi']

# colors = ['violet','orangered','royalblue','orange','darkviolet','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive']
# var_list = ['e_s','rh','vpd','wspd','soil_m','hdw','haines','kbdi','ffwi','mffwi','fwi',"fm100","fm1000","erc","bi"]
colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive','lightgray', 'darkcyan', 'darkred', 'green']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]


# var_list_short = ['e_s','vpd','fwi','soil_m','fm100','fm1000']

# ax[2,0].set_position([0.0,0,1.0,0.25])

ax[0,1].set_position([0.45,0.47,0.25,0.36])
ax[0,2].set_position([0.75,0.47,0.25,0.36])
ax[1,1].set_position([0.45,0,0.25,0.36])
ax[1,2].set_position([0.75,0.,0.25,0.36])

ax[0,0].set_position([0.,0.47,0.36,0.36])
ax[1,0].set_position([0.,0.,0.36,0.36])

# ax[2,1].set_position([0.45,-0.47,0.25,0.36])
# ax[2,2].set_position([0.75,-0.47,0.25,0.36])

# ax[2,0].set_position([0.,-0.47,0.38,0.36])

row = 0
for vegetation in ['forest','non_forest']:
    df_tmp = eval(f'FIRED_{vegetation}')
    mean_auc_list = []
    
    count_cdf_dict[vegetation] = {}
    count_pdf_dict[vegetation] = {}
    
    tick_color_list = []
    i = 0
    # for var in var_list_short:
    for var in var_list:
        if i < 6:
            col = 1
        else:
            col = 2

        mean = np.zeros(100)

        count_cdf_dict[vegetation][var] = pd.DataFrame()
        count_pdf_dict[vegetation][var] = pd.DataFrame()

        mean_entropy = 0
        data = pd.DataFrame()
        for pyrome_id in sorted(pyrome['PYROME']):
            tmp = df_tmp[(df_tmp['PYROME'] == pyrome_id) & (df_tmp['event_day'] == 1)][['ig_date','event_day','id']]
            tmp['fwi'] = era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = tmp['ig_date'].values).values
            data = pd.concat([data,tmp])

        count_pdf_dict[vegetation][var][pyrome_id] = np.histogram(1 - data['fwi'], bins = 100, range = (0,1), density = True)[0]
        count_cdf_dict[vegetation][var][pyrome_id] = np.cumsum(np.histogram(1 - data['fwi'], bins = 100, range = (0,1), density = True)[0])

        mean_auc_list += [np.sum(count_cdf_dict[vegetation][var].median(axis = 1) - np.linspace(1,100,100))/50/100]

        ax[row,col].plot(count_cdf_dict[vegetation][var].median(axis = 1), color = colors[i], linewidth = 2.5,
                         label = f'{var.upper()}')


        for j in np.arange(0,101,10):
            ax[row,col].axhline(j, c = 'grey', linewidth = 0.1, linestyle = '--')
        for j in np.arange(0,101,10):
            ax[row,col].axvline(j, c = 'grey', linewidth = 0.1, linestyle = '--')
        ax[row,col].plot(range(0,100),range(0,100), c = 'grey', linestyle = '--')

        if i in [0,6]:
            for j in np.arange(0,101,10):
                ax[row,col].axhline(j, c = 'grey', linewidth = 0.1, linestyle = '--')
            for j in np.arange(0,101,10):
                ax[row,col].axvline(j, c = 'grey', linewidth = 0.1, linestyle = '--')
        #     ax[col].plot(range(0,100),range(0,100), c = 'grey', linestyle = '--')

            ax[row,col].fill_between(range(0,100),range(0,100),100*np.ones(100), color = 'cornflowerblue', alpha = 0.12, linestyle = '--')
            ax[row,col].fill_between(range(0,100),np.zeros(100),range(0,100), color = 'sandybrown', alpha = 0.35, linestyle = '--')

            ax[row,col].set_xticks([0,25,50,75,100])
            ax[row,col].set_xticklabels([100,75,50,25,0], fontsize = 14)
        
        if row == 1:
            if col == 1:
                ax[row,col].legend(loc = 'lower right', fontsize = 16, handlelength = 1.3)
            else:
                ax[row,col].legend(loc = [1.05,0], fontsize = 20, handlelength = 1.3)
            
        if col == 0:
            tick_color_list += ['blue']
        else:
            tick_color_list += ['black']
        i += 1

    auc_sorted = [i[0] for i in sorted(zip(mean_auc_list,colors,var_list,tick_color_list),key=lambda x:x[0])]
    color_sorted = [i[1] for i in sorted(zip(mean_auc_list,colors,var_list,tick_color_list),key=lambda x:x[0])]
    var_sorted = [i[2].upper() for i in sorted(zip(mean_auc_list,colors,var_list,tick_color_list),key=lambda x:x[0])]    

    for threshold in np.arange(-0.2,0.61,0.1):
        ax[row,0].axhline(threshold, linestyle = '--', linewidth = 0.5, c= 'grey')
    ax[row,0].axhline(0, linestyle = '-', linewidth = 1.5, c = 'black')
    
    p = ax[row,0].bar(range(len(mean_auc_list)),auc_sorted, color = color_sorted, alpha = 0.8)
    ax[row,0].bar_label(p, labels=[round(i,2) for i in auc_sorted], label_type='edge',fontsize = 12, rotation = -40, padding = 3)

    
    
    ax[row,0].set_xticks(range(len(mean_auc_list)))
    ax[row,0].set_xticklabels(var_sorted,rotation = 80,fontsize = 17)
    
    ax[row,0].set_yticks(np.arange(-0.2,0.61,0.2))
    ax[row,0].set_yticklabels([round(i,1) for i in np.arange(-0.2,0.61,0.2)],fontsize = 17)
    ax[row,0].set_ylabel('AUC score',fontsize = 23)
    
    for col in range(1,3):
        ax[row,col].set_yticks(np.arange(0,101,20))
        ax[row,col].set_yticklabels([round(i,1) for i in np.arange(0,101,20)],fontsize = 17)

        ax[row,col].set_xticks(np.arange(0,101,20))
        ax[row,col].set_xticklabels([round(i,1) for i in np.arange(0,101,20)],fontsize = 17)

    for ticklabel in ax[row,0].get_xticklabels():
        if ticklabel.get_text() in [i.upper() for i in ['e_s','rh','vpd','wspd','wspdmax','soil_m']]:
            ticklabel.set_color('blue')
        if ticklabel.get_text() in [i.upper() for i in ['wspdmax']]:
            ticklabel.set_fontsize(15)
    
    # ax[row,0].tick_params(axis='x', colors=tick_color_list)
    ax[row,0].set_ylim([-0.32,0.68])
    ax[row,1].set_ylabel('Percent of Ignitions Captured', fontsize = 20, labelpad = 5)
  
    row += 1

for col in range(1,3):
    ax[1,col].text(0.5,-0.15,'Index Percentile', fontsize = 21,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[1,col].transAxes)

ax[0,0].text(-0.2,0.5,f'Forest\n(n = {len(FIRED_forest[FIRED_forest['event_day'] == 1])} fires)', fontsize = 26,
            horizontalalignment='right',
            verticalalignment='center', transform=ax[0,0].transAxes)
ax[1,0].text(-0.2,0.5,f'Non-Forest\n(n = {len(FIRED_non_forest[FIRED_non_forest['event_day'] == 1])} fires)', fontsize = 26,
            horizontalalignment='right',
            verticalalignment='center', transform=ax[1,0].transAxes)
# ax[2,0].text(-0.2,0.5,'Grass\n& Shrub', fontsize = 18,
#             horizontalalignment='right',
#             verticalalignment='center', transform=ax[2,0].transAxes)

ax[0,0].text(-0.09,1.10,'a', fontsize = 32,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[0,0].transAxes)
ax[0,1].text(-0.12,1.10,'b', fontsize = 32,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[0,1].transAxes)
ax[0,2].text(-0.12,1.10,'c', fontsize = 32,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[0,2].transAxes)
ax[1,0].text(-0.09,1.10,'d', fontsize = 32,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[1,0].transAxes)
ax[1,1].text(-0.12,1.10,'e', fontsize = 32,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[1,1].transAxes)
ax[1,2].text(-0.12,1.10,'f', fontsize = 32,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[1,2].transAxes)

ax[0,1].set_title('Hydrometeorological\nVariables', fontsize = 29, y = 1.02)
ax[0,2].set_title('Fire Indices', fontsize = 29, y = 1.05)


In [ ]:
fig,ax = plt.subplots(4,3,figsize = (16,20), dpi = 400, sharex = False, sharey = False)

# var_list = ['e','e_s','rh','vpd','uv10','soil_m','hdw','haines','FFWI','kbdi','mffwi','fwi']

colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive', 'lightgray', 'darkcyan', 'darkred', 'green']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]

# var_list_short = ['e_s','vpd','fwi','soil_m','fm100','fm1000']

# ax[2,0].set_position([0.0,0,1.0,0.25])

ax[0,1].set_position([0.45,0.75,0.25,0.2])
ax[0,2].set_position([0.75,0.75,0.25,0.2])
ax[1,1].set_position([0.45,0.5,0.25,0.2])
ax[1,2].set_position([0.75,0.5,0.25,0.2])

ax[0,0].set_position([0.,0.75,0.38,0.2])
ax[1,0].set_position([0.,0.5,0.38,0.2])

ax[2,1].set_position([0.45,0.25,0.25,0.2])
ax[2,2].set_position([0.75,0.25,0.25,0.2])
ax[3,1].set_position([0.45,0,0.25,0.2])
ax[3,2].set_position([0.75,0.,0.25,0.2])

ax[2,0].set_position([0.,0.25,0.38,0.2])
ax[3,0].set_position([0.,0.,0.38,0.2])
# ax[2,1].set_position([0.45,-0.47,0.25,0.36])
# ax[2,2].set_position([0.75,-0.47,0.25,0.36])

# ax[2,0].set_position([0.,-0.47,0.38,0.36])

row = 0
for vegetation in ['forest','savanna','shrub','grass']:
    df_tmp = eval(f'FIRED_{vegetation}')
    mean_auc_list = []
    
    count_cdf_dict[vegetation] = {}
    count_pdf_dict[vegetation] = {}
    
    tick_color_list = []
    i = 0
    # for var in var_list_short:
    for var in var_list:
        if i < 5:
            col = 1
        else:
            col = 2

        mean = np.zeros(100)

        count_cdf_dict[vegetation][var] = pd.DataFrame()
        count_pdf_dict[vegetation][var] = pd.DataFrame()

        mean_entropy = 0
        data = pd.DataFrame()
        for pyrome_id in sorted(pyrome['PYROME']):
            tmp = df_tmp[(df_tmp['PYROME'] == pyrome_id) & (df_tmp['event_day'] == 1)][['ig_date','event_day','id']]
            tmp['fwi'] = era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = tmp['ig_date'].values).values
            data = pd.concat([data,tmp])

        count_pdf_dict[vegetation][var][pyrome_id] = np.histogram(1 - data['fwi'], bins = 100, range = (0,1), density = True)[0]
        count_cdf_dict[vegetation][var][pyrome_id] = np.cumsum(np.histogram(1 - data['fwi'], bins = 100, range = (0,1), density = True)[0])

        mean_auc_list += [np.sum(count_cdf_dict[vegetation][var].median(axis = 1) - np.linspace(1,100,100))/50/100]

        ax[row,col].plot(count_cdf_dict[vegetation][var].median(axis = 1), color = colors[i], linewidth = 2.5,
                         label = f'{var.upper()}')


        for j in np.arange(0,101,10):
            ax[row,col].axhline(j, c = 'grey', linewidth = 0.1, linestyle = '--')
        for j in np.arange(0,101,10):
            ax[row,col].axvline(j, c = 'grey', linewidth = 0.1, linestyle = '--')
        ax[row,col].plot(range(0,100),range(0,100), c = 'grey', linestyle = '--')

        if i in [0,5]:
            for j in np.arange(0,101,10):
                ax[row,col].axhline(j, c = 'grey', linewidth = 0.1, linestyle = '--')
            for j in np.arange(0,101,10):
                ax[row,col].axvline(j, c = 'grey', linewidth = 0.1, linestyle = '--')
        #     ax[col].plot(range(0,100),range(0,100), c = 'grey', linestyle = '--')

            ax[row,col].fill_between(range(0,100),range(0,100),100*np.ones(100), color = 'cornflowerblue', alpha = 0.12, linestyle = '--')
            ax[row,col].fill_between(range(0,100),np.zeros(100),range(0,100), color = 'sandybrown', alpha = 0.35, linestyle = '--')

            ax[row,col].set_xticks([0,25,50,75,100])
            ax[row,col].set_xticklabels([100,75,50,25,0])
        
        if row == 3:
            if col == 1:
                ax[row,col].legend(loc = 'lower right', fontsize = 16, handlelength = 1.3)
            else:
                ax[row,col].legend(loc = [1.05,0], fontsize = 18, handlelength = 1.3)
                
        if col == 0:
            tick_color_list += ['blue']
        else:
            tick_color_list += ['black']
        i += 1

    auc_sorted = [i[0] for i in sorted(zip(mean_auc_list,colors,var_list,tick_color_list),key=lambda x:x[0])]
    color_sorted = [i[1] for i in sorted(zip(mean_auc_list,colors,var_list,tick_color_list),key=lambda x:x[0])]
    var_sorted = [i[2].upper() for i in sorted(zip(mean_auc_list,colors,var_list,tick_color_list),key=lambda x:x[0])]    

    for threshold in np.arange(-0.2,0.61,0.1):
        ax[row,0].axhline(threshold, linestyle = '--', linewidth = 0.5, c= 'grey')
    ax[row,0].axhline(0, linestyle = '-', linewidth = 1.5, c = 'black')
    
    p = ax[row,0].bar(range(len(mean_auc_list)),auc_sorted, color = color_sorted, alpha = 0.8)
    ax[row,0].bar_label(p, labels=[round(i,2) for i in auc_sorted], label_type='edge',fontsize = 11, rotation = -40, padding = 1)

    
    
    ax[row,0].set_xticks(range(len(mean_auc_list)))
    ax[row,0].set_xticklabels(var_sorted,rotation = 70,fontsize = 14)
    
    ax[row,0].set_yticks(np.arange(-0.2,0.61,0.2))
    ax[row,0].set_yticklabels([round(i,1) for i in np.arange(-0.2,0.61,0.2)],fontsize = 16)
    ax[row,0].set_ylabel('AUC score',fontsize = 18)
    
    for col in range(1,3):
        ax[row,col].set_yticks(np.arange(0,101,20))
        ax[row,col].set_yticklabels([round(i,1) for i in np.arange(0,101,20)],fontsize = 16)

        ax[row,col].set_xticks(np.arange(0,101,20))
        ax[row,col].set_xticklabels([round(i,1) for i in np.arange(0,101,20)],fontsize = 16)

    for ticklabel in ax[row,0].get_xticklabels():
        if ticklabel.get_text() in [i.upper() for i in ['e_s','rh','vpd','wspd','wspdmax','soil_m']]:
            ticklabel.set_color('blue')
    
    # ax[row,0].tick_params(axis='x', colors=tick_color_list)
    ax[row,0].set_ylim([-0.32,0.68])
    ax[row,1].set_ylabel('Percent of Ignitions Captured', fontsize = 20, labelpad = 0)
  
    row += 1

for col in range(1,3):
    ax[3,col].text(0.5,-0.15,'Index Percentile', fontsize = 25,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[3,col].transAxes)

ax[0,0].text(-0.2,0.5,f'Forest\n(n = {len(FIRED_forest[FIRED_forest['event_day'] == 1])} fires)', fontsize = 25,
            horizontalalignment='right',
            verticalalignment='center', transform=ax[0,0].transAxes)
ax[1,0].text(-0.2,0.5,f'Savanna\n(n = {len(FIRED_savanna[FIRED_savanna['event_day'] == 1])} fires)', fontsize = 25,
            horizontalalignment='right',
            verticalalignment='center', transform=ax[1,0].transAxes)
ax[2,0].text(-0.2,0.5,f'Shrub\n(n = {len(FIRED_shrub[FIRED_shrub['event_day'] == 1])} fires)', fontsize = 25,
            horizontalalignment='right',
            verticalalignment='center', transform=ax[2,0].transAxes)
ax[3,0].text(-0.2,0.5,f'Grass\n(n = {len(FIRED_grass[FIRED_grass['event_day'] == 1])} fires)', fontsize = 25,
            horizontalalignment='right',
            verticalalignment='center', transform=ax[3,0].transAxes)


ax[0,1].set_title('Hydrometeorological\nVariables', fontsize = 26, y = 1.02)
ax[0,2].set_title('Fire Indices', fontsize = 26, y = 1.02)

i = 0
letter_list = ['a','b','c','d','e','f','g','h','i','j','k','l','m']
for row in range(4):
    for col in range(3):
        ax[row,col].text(-0.09,1.10,letter_list[i], fontsize = 32,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[row,col].transAxes)
        i += 1


In [ ]:
fig,ax=plt.subplots(2,4,figsize=(16,9), dpi = 250, subplot_kw={'projection': ccrs.PlateCarree()})

# pyrome = gpd.read_file('data/Pyromes/Pyromes_lon105W.shp')
# pyrome.index = pyrome['PYROME'].values

# land_mask = xr.where(era5_wrf.SOIL_M.max(dim = 'day') > 0, 1, 0)
# land_mask = land_mask.load()

i = 1
best_var_list = ['e_s','vpd','dc','kbdi','fm1000','soil_m']

for var in best_var_list:

    pyrome[f'{var}_auc'] = auc_df[var]

    row,col = i//4,i%4
    if row == 1:
        col += 1

    bounds = list(np.arange(0.,.81,0.1))
    cmap = cm.get_cmap('Reds', 8)
    colors = cmap(np.arange(0,cmap.N))
    norm = matplotlib.colors.BoundaryNorm(boundaries=bounds, ncolors = 8)

    pyrome.plot(ax = ax[row,col], column = f'{var}_auc', cmap = cmap, norm = norm, edgecolor = 'k', linewidth = 1)
    # p = pyrome.plot(ax = ax, column = 'soil_m_entropy', edgecolor = 'k', linewidth = 1)
    ax[row,col].set_title(var.upper(), fontsize = 23, y = 1.02)
    if col == 0:
        ax[row,col].set_position([col*0.23,0.5-row*0.5,0.21,0.43])
    else:
        ax[row,col].set_position([col*0.23+0.08,0.5-row*0.5,0.21,0.43])
    
    i += 1
 
cax = fig.add_axes([1, 0.03, 0.025, 0.9])
sm = plt.cm.ScalarMappable(cmap = cmap, norm = norm)
sm._A = []
cbar = fig.colorbar(sm, cax=cax, extend = 'both')

cbar.set_ticks(np.arange(0,0.81,0.1))
cbar.set_ticklabels([round(i,1) for i in cbar.get_ticks()], fontsize = 18)
cbar.set_label('AUC score', rotation = 270, fontsize = 27, labelpad = 40)

##### Finding best variable for each pyrome
tmp = pyrome[[i for i in pyrome.columns if i.endswith('auc')]].idxmax(axis = 1)
tmp[:] = [i[:-4] for i in tmp]

i = 1
for var in best_var_list:
    tmp[:] = np.where(tmp == var, i, tmp)
    i += 1
pyrome['best_var'] = [int(i) for i in tmp]


###### Plotting for best variable in each pyrome
bounds = list(np.arange(0.5,6.6,1))
cmap = ListedColormap(['palegoldenrod','gold','hotpink','lightskyblue','dodgerblue','royalblue'])
colors = cmap(np.arange(0,6))
norm = matplotlib.colors.BoundaryNorm(boundaries=bounds, ncolors = 6)

pyrome.plot(ax = ax[0,0], column = f'best_var', cmap = cmap, norm = norm, vmin = 0.5, vmax = 6.5, edgecolor = 'k', linewidth = 1)
cax = fig.add_axes([0.23, 0.5, 0.015, 0.45])
sm = plt.cm.ScalarMappable(cmap = cmap, norm = norm)
sm._A = []
cbar = fig.colorbar(sm, cax=cax)

cbar.set_ticks(np.arange(1,6.1,1))
cbar.set_ticklabels([i.upper() for i in best_var_list], fontsize = 15)

ax[0,0].set_position([0.0,0.5,0.21,0.45])

ax[0,0].set_title('Best Variable for Pyrome', fontsize = 20, y = 1.02)

i = 0
letter_list = ['a','b','c','d','e','f','g','h','i','j','k','l','m']
for row in range(2):
    for col in range(4):
        if i > 0:
            ax[row,col].text(-0.06,1.08,letter_list[i], fontsize = 26,
                horizontalalignment='center',
                verticalalignment='center', transform=ax[row,col].transAxes)
        
        else: 
            ax[row,col].text(-0.12,1.04,letter_list[i], fontsize = 26,
                horizontalalignment='center',
                verticalalignment='center', transform=ax[row,col].transAxes)
        if (row == 1) & (col == 0):
            pass
        else:
            i += 1
        
plt.delaxes(ax[1,0])

In [ ]:
# fig,ax = plt.subplots(6,4,figsize = (10,4), dpi = 100, sharex = True, sharey = True)

colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive','lightgray', 'darkcyan', 'darkred', 'green']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','ffmc','dmc','dc','isi','bui','fwi',"fm100","fm1000","erc","bi"]

i = 0

spread_dict = {}
# count_dict = {}

for var in var_list:
    print(var)
    row, col = i //4, i % 4
    spread_dict[var] = pd.DataFrame()
#     count_dict[var] = pd.DataFrame()
    for pyrome_id in sorted(pyrome['PYROME']):
        spread_list = []
#         count_list = []
        data = FIRED[(FIRED['PYROME'] == pyrome_id)][['ig_date','event_day','date','dy_ar_km2','id','month']]
        # data = data[(data['month'] >= 6) & (data['month'] <= 10)]

        # data = data[(data['dy_ar_km2'] >= 5)]
        data['fwi'] = era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = data['date'].values).rank('day',pct = True)
        # data['fwi'] = era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = data['date'].values)

        for threshold in np.arange(99,-1,-1):
            # spread_list += [(np.sum(data[data['fwi']*100 >= threshold]['dy_ar_km2']))/np.sum(data['dy_ar_km2'])]
            spread_list += [(np.mean(data[data['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(data['dy_ar_km2'])]
            # spread_list += [np.mean(data[data['fwi']*100 >= threshold]['dy_ar_km2']/(data[data['fwi']*100 >= threshold]['tot_ar_km2'] - data[data['fwi']*100 >= threshold]['dy_ar_km2']))]
#             count_list += [len(data[data['fwi']*100 >= threshold])]
        
        spread_dict[var][pyrome_id] = spread_list

    i += 1

In [ ]:
# land_mask = xr.where(era5_wrf.SOIL_M.max(dim = 'day') > 0, 1, 0)
# land_mask = land_mask.load()
fig,ax = plt.subplots(1,3,figsize = (11,3), dpi = 400, sharex = False, sharey = False)

# ax[0]=plt.subplot(1,1,sharex=True,sharey=True,figsize=(4,6), dpi = 250, subplot_kw={'projection': ccrs.PlateCarree()})
# ax[0]=plt.subplot(1,1,1)
    
bounds = list(range(0,51,5))
cmap = cm.get_cmap('Reds', 10)
colors = cmap(np.arange(0,cmap.N))
norm = matplotlib.colors.BoundaryNorm(boundaries=bounds, ncolors=10)

p = pyrome.plot(ax = ax[0], column = 'fire_count_per_year', cmap = cmap, norm = norm, edgecolor = 'k', linewidth = 1)
# p = pyrome.plot(ax = ax, column = 'region', cmap = cmap, vmin = 0, vmax = 4, edgecolor = 'k', linewidth = 1)
# p = pyrome.plot(ax = ax, column = 'forest_fire_count_per_year', cmap = cmap, vmin = 0, vmax = 10, edgecolor = 'k', linewidth = 1)

cax = fig.add_axes([0.27, 0.05, 0.02, 0.9])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm._A = []
cbar = fig.colorbar(sm, cax=cax, extend = 'max')
cbar.set_label('Number of fires per year (2001-2022)', rotation = 270, labelpad = 15, fontsize = 12)

# ax[0].set_title('Pyromes in western US', fontsize = 15)
ax[0].set_xlabel('Longitude', fontsize = 12)
ax[0].set_ylabel('Latitude', fontsize = 12)

# var_list = ['e','e_s','rh','vpd','uv10','soil_m','hdw','haines','FFWI','kbdi','mffwi','fwi']

i = 0

var = 'vpd'
pyrome_id = 20
# for var in var_list:
# for pyrome_id in sorted(pyrome['PYROME']):
# plt.boxplot([era5_wrf_basin_pct_rank_dict[var].sel(basin = pyrome_id).sel(day = data['ignition_d'].values).values for var in var_list])
#     row, col = i // 4, i % 4
#     print(var)
ax[1].plot(range(100),count_cdf_dict[var][pyrome_id], color = 'royalblue', linewidth = 3)
ax[1].fill_between(range(100),count_cdf_dict[var][pyrome_id],range(100), color = 'cornflowerblue', alpha = 0.7)
ax[1].fill_between(range(100),range(100),np.zeros(100), color = 'sandybrown', alpha = 0.7)
# ax[0].hist(1 - era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = data['ignition_d'].values).values, bins = 100, range = (0,1), alpha = 0.3, density = True, cumulative = True)
# ax[0].hist(1 - era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).values, bins = 100, range = (0,1), alpha = 0.5, density = True, cumulative = True)
# ax[0].set_title(var.upper())
for j in np.arange(0,101,10):
    ax[1].axhline(j, c = 'grey', linewidth = 0.2, linestyle = '--')
for j in np.arange(0,101,10):
    ax[1].axvline(j, c = 'grey', linewidth = 0.2, linestyle = '--')

ax[1].plot(range(0,100),range(0,100), c = 'grey', linestyle = '-')
# ax[0].set_title('Southern Sierra Nevada - VPD', fontsize = 14)
# ax[0].set_title('Southern California Mountains - VPD', fontsize = 14)
ax[1].set_ylabel('Percent of Ignitions Captured', fontsize = 12)
ax[1].set_xlabel('Index Percentile', fontsize = 12)

ax[1].set_xticks(range(0,101,20))
ax[1].set_xticklabels(range(100,-1,-20))
# for j in np.arange(0,1.01,0.2):
#     ax[0].axhline(j, c = 'grey', linewidth = 0.2, linestyle = '--')
# for j in np.arange(0,1.01,0.2):
#     ax[0].axvline(j, c = 'grey', linewidth = 0.2, linestyle = '--')

# var = 'vpd'
for threshold in np.arange(0,7,1):
    if threshold == 1:
        ax[2].axhline(threshold, linewidth = 0.7, color = 'grey', linestyle = '-')
    else:
        ax[2].axhline(threshold, linewidth = 0.2, color = 'grey', linestyle = '--')
        
# for threshold in np.arange(0,101,25):
#     ax[1].axvline(threshold, linewidth = 0.2, color = 'grey', linestyle = '--')
# ax[i].fill_between()    
ax[2].fill_between(range(100),np.ones(100),np.zeros(100), color = 'sandybrown', alpha = 0.7)

ax[2].set_xticks([0,25,50,75,100])
ax[2].set_xticklabels([100,75,50,25,0])
ax[2].set_xlabel('Index Percentile', fontsize = 12)
    
ax[2].set_ylabel('Fire Spread Relative to Pyrome Mean',fontsize =12,labelpad = 10)

# ax[1].set_title('Southern Sierra Nevada\nVPD')
ax[2].plot(range(100),spread_dict[var][pyrome_id],color = 'royalblue',linewidth = 3)
ax[2].fill_between(range(100),spread_dict[var][pyrome_id],np.ones(100), color = 'cornflowerblue', alpha = 0.7)

ax[0].set_position([0.,0,0.28,1])
ax[1].set_position([0.42,0,0.28,1])
ax[2].set_position([0.77,0,0.22,1])

# i += 1
# plt.boxplot([2]*190,)
#     ax[row,col].set_xticklabels([i.upper() for i in var_list])
# plt.boxplot([1],era5_wrf_basin_pct_rank_dict['fwi'].sel(basin = pyrome_id).sel(day = data['ignition_d'].values))
# plt.delaxes(ax[2,3])

ax[0].text(-0.09,1.08,'a', fontsize = 22,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[0].transAxes)
ax[1].text(-0.05,1.08,'b', fontsize = 22,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[1].transAxes)
ax[2].text(-0.05,1.08,'c', fontsize = 22,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[2].transAxes)


ax[1].annotate("Less skill", xytext=(35, 53), xy=(31,69),
            arrowprops=dict(arrowstyle="<-"), fontsize = 13)
ax[1].annotate("More skill", xytext=(0, 85), xy=(30,70),
            arrowprops=dict(arrowstyle="<-"), fontsize = 13)

ax[2].annotate("More skill", xytext=(12, 3.3), xy=(20,2.2), fontsize = 13,
        arrowprops=dict(arrowstyle="<-"))
ax[2].annotate("Less skill", xytext=(6, 1.1), xy=(20,2.2),
            arrowprops=dict(arrowstyle="<-"), fontsize = 13)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np

# -------------------------------
# VARIABLES & COLORS
# -------------------------------
colors = [
    'violet','orangered','royalblue','orange','peru','pink','magenta',
    'olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue',
    'navy','limegreen','sienna','olive','lightgray','darkcyan','darkred','green'
]

var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m',
            'hdw','haines','kbdi','ffwi','mffwi',
            'ffmc','dmc','dc','isi','bui','fwi',
            "fm100","fm1000","erc","bi"]


# -------------------------------
# FIGURE LAYOUT
# -------------------------------
n_top = 6
n_bottom = len(var_list) - n_top
n_cols = max(n_top, n_bottom)

fig = plt.figure(figsize=(15, 15), dpi=200)
gs = GridSpec(
    nrows=3,
    ncols=n_cols,
    height_ratios=[1, 1, 0.7],
    hspace=0.35
)

# create axes for variable plots
axes = []
for i in range(len(var_list)):
    if i < n_top:
        ax = fig.add_subplot(gs[0, i])
    else:
        ax = fig.add_subplot(gs[1, i - n_top])
    axes.append(ax)

# -------------------------------
# MAIN PLOTTING LOOP
# -------------------------------
spread_mean_auc_list = []
auc_std_list = []

for i, var in enumerate(var_list):
    ax = axes[i]

    var_auc_list = []
    for pyrome_id in sorted(pyrome['PYROME']):
        ax.plot(
            spread_dict[var][pyrome_id],
            c='royalblue',
            linewidth=0.08,
            alpha=0.7
        )
        var_auc_list.append(np.sum(spread_dict[var][pyrome_id]) / 100 - 1)

    mean_curve = spread_dict[var].mean(axis=1)
    ax.plot(
        mean_curve,
        color='k',
        label=round(mean_curve.sum() / 100, 2)
    )

    spread_mean_auc_list.append(mean_curve.sum() / 100 - 1)
    auc_std_list.append(np.std(var_auc_list))

    for percentile in [0.25, 0.75]:
        ax.plot(
            spread_dict[var].quantile(percentile, axis=1),
            linewidth=0.75,
            color='orange'
        )

    for threshold in np.arange(0, 3.5, 0.25):
        if threshold == 1:
            ax.axhline(threshold, linewidth=0.7, color='grey')
        else:
            ax.axhline(threshold, linewidth=0.2, color='grey', linestyle='--')

    ax.set_ylim([0, 3.5])
    ax.set_title(var.upper(), fontsize=12.5)
    # if var == 'wspdmax':
    #     ax.set_title(var.upper(), fontsize=13)

    ax.set_xticks([0, 50, 100])
    ax.set_xticklabels([100, 50, 0], rotation=80, fontsize=11)
    
    ax.set_yticks(np.arange(0,3.6,0.5))
    if i in [0,6]:
        ax.set_yticklabels(np.arange(0,3.6,0.5))
    else:
        ax.set_yticklabels([])
        
    ax.text(
        0.65, 0.95,
        f'{round(mean_curve.sum()/100 - 1, 2)}',
        fontsize=10,
        transform=ax.transAxes,
        ha='center',
        va='center'
    )

# -------------------------------
# SHARED LABELS & ANNOTATIONS
# -------------------------------
axes[0].set_ylabel('Fire Spread Relative to Pyrome Mean', fontsize=13, labelpad = 10)
axes[6].set_ylabel('Fire Spread Relative to Pyrome Mean', fontsize=13, labelpad = 10)
axes[2].set_xlabel('Index Percentile', x = 1.1, fontsize=14)
axes[13].set_xlabel('Index Percentile', fontsize=16)

axes[2].text(
    1.1, 1.12, 'Hydrometeorological Variables',
    fontsize=18, transform=axes[2].transAxes,
    ha='center'
)

axes[13].text(
    0.5, 1.12, 'Fire Indices',
    fontsize=18, transform=axes[13].transAxes,
    ha='center'
)

axes[0].text(
    -0.35, 1.15, 'a',
    fontsize=22, transform=axes[0].transAxes
)

# -------------------------------
# BAR PLOT (BOTTOM)
# -------------------------------
ax1 = fig.add_subplot(gs[2, :])

auc_sorted = [i[0] for i in sorted(
    zip(spread_mean_auc_list, colors, var_list, auc_std_list),
    key=lambda x: x[0]
)]
color_sorted = [i[1] for i in sorted(
    zip(spread_mean_auc_list, colors, var_list, auc_std_list),
    key=lambda x: x[0]
)]
var_sorted = [i[2].upper() for i in sorted(
    zip(spread_mean_auc_list, colors, var_list, auc_std_list),
    key=lambda x: x[0]
)]
auc_std_list_sorted = [i[3] for i in sorted(
    zip(spread_mean_auc_list, colors, var_list, auc_std_list),
    key=lambda x: x[0]
)]

for threshold in np.arange(-0.1, 0.66, 0.05):
    ax1.axvline(threshold, linestyle='--', linewidth=0.5, c='grey')
ax1.axvline(0, linestyle='-', linewidth=1.5, c='black')

p = ax1.barh(
    range(len(auc_sorted)),
    auc_sorted,
    color=color_sorted,
    alpha=0.8
)

ax1.bar_label(
    p,
    labels=[
        f'{round(auc_sorted[i],2)}±{round(auc_std_list_sorted[i],2)}'
        for i in range(len(auc_sorted))
    ],
    label_type='edge',
    fontsize=10,
    padding=5
)

ax1.set_yticks(range(len(var_sorted)))
ax1.set_yticklabels(var_sorted, fontsize=12)
ax1.yaxis.tick_right()
ax1.yaxis.set_ticks_position('both')

ax1.set_xticks(np.arange(-0.1, 0.51, 0.1))
ax1.set_xticklabels(
    [round(i, 1) for i in np.arange(-0.1, 0.51, 0.1)],
    fontsize=12
)
ax1.set_xlabel('AUC score', fontsize=13)
ax1.set_xlim([-0.05, 0.6])

for ticklabel in ax1.get_yticklabels():
    if ticklabel.get_text() in [i.upper() for i in ['e_s','rh','vpd','wspd','wspdmax','soil_m']]:
        ticklabel.set_color('blue')
ax1.set_position([0.1,0.03,0.8,0.25])

ax1.text(
    -0.04, 1.05, 'b',
    fontsize=22, transform=ax1.transAxes
)

plt.show()


In [ ]:

# var_list = ['e','e_s','rh','vpd','uv10','soil_m','hdw','haines','FFWI','kbdi','mffwi','fwi']
# var_list = ['e','e_s','rh','vpd','uv10','soil_m','hdw','fwi','FFWI','mffwi','haines','kbdi',"fm1","fm10","fm100","fm1000"]
# var_list = ['e','e_s','rh','vpd','wspd','soil_m','hdw','fwi','ffwi','mffwi','haines','kbdi',"fm1","fm10","fm100","fm1000","erc","bi","sc","ic","ros"]
# var_list = ['e_s','rh','vpd','uv10','soil_m','hdw','haines','kbdi','ffwi','mffwi','fwi',"fm100","fm1000","erc","bi"]

colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive','teal', 'darkcyan', 'darkred', 'green', 'lightgray']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]

i = 0

spread_anomaly_dict = {}
# count_dict = {}

for var in var_list:
    print(var)
    row, col = i //4, i % 4
    spread_anomaly_dict[var] = pd.DataFrame()
#     count_dict[var] = pd.DataFrame()
    for pyrome_id in sorted(pyrome['PYROME']):
        spread_list = []
#         count_list = []
        data = FIRED[(FIRED['PYROME'] == pyrome_id)][['ig_date','event_day','date','dy_ar_km2','id']]
        # data = data[(data['dy_ar_km2'] >= 5)]
        data['fwi'] = era5_wrf_basin_anomaly_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = data['date'].values).rank('day',pct = True)

        for threshold in np.arange(99,-1,-1):
            # spread_list += [(np.sum(data[data['fwi']*100 >= threshold]['dy_ar_km2']))/np.sum(data['dy_ar_km2'])]
            spread_list += [(np.mean(data[data['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(data['dy_ar_km2'])]
            # spread_list += [np.mean(data[data['fwi']*100 >= threshold]['dy_ar_km2']/(data[data['fwi']*100 >= threshold]['tot_ar_km2'] - data[data['fwi']*100 >= threshold]['dy_ar_km2']))]
#             count_list += [len(data[data['fwi']*100 >= threshold])]
        
        spread_anomaly_dict[var][pyrome_id] = spread_list

    i += 1
    
i = 0

spread_season_dict = {}
# count_dict = {}

for var in var_list:
    print(var)
    row, col = i //4, i % 4
    spread_season_dict[var] = pd.DataFrame()
#     count_dict[var] = pd.DataFrame()
    for pyrome_id in sorted(pyrome['PYROME']):
        spread_list = []
#         count_list = []
        data = FIRED[(FIRED['PYROME'] == pyrome_id)][['ig_date','event_day','date','dy_ar_km2','id']]
        # data = data[(data['dy_ar_km2'] >= 5)]
        data['fwi'] = era5_wrf_basin_season_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = data['date'].values).rank('day',pct = True)

        for threshold in np.arange(99,-1,-1):
            # spread_list += [(np.sum(data[data['fwi']*100 >= threshold]['dy_ar_km2']))/np.sum(data['dy_ar_km2'])]
            spread_list += [(np.mean(data[data['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(data['dy_ar_km2'])]
            # spread_list += [np.mean(data[data['fwi']*100 >= threshold]['dy_ar_km2']/(data[data['fwi']*100 >= threshold]['tot_ar_km2'] - data[data['fwi']*100 >= threshold]['dy_ar_km2']))]
#             count_list += [len(data[data['fwi']*100 >= threshold])]
        
        spread_season_dict[var][pyrome_id] = spread_list

    i += 1

In [ ]:
# fig,ax = plt.subplots(3,1,figsize = (8,9), dpi = 400, sharex = True)
fig,ax = plt.subplots(1,3,figsize = (14,4.5), dpi = 400, sharex = True)

colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive','teal', 'darkcyan', 'darkred', 'green', 'lightgray']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]

i = 0
for var in var_sorted[::-1]:
    var = var.lower()
    var_auc_list = []
    for pyrome_id in sorted(pyrome['PYROME']):
        var_auc_list += [np.sum(spread_dict[var][pyrome_id] - 1)/100]
    p = ax[0].barh(len(var_list) - i, np.mean(var_auc_list), color = color_sorted[::-1][i], height = 0.75)
    if np.mean(var_auc_list) > 0:
        ax[0].bar_label(p, labels=[f'{round(np.mean(var_auc_list),2)}'], label_type='edge',fontsize = 13, padding = 5)
    else:
        ax[0].text(0.01,len(var_list) - i,f'{round(np.mean(var_auc_list),2)}', fontsize = 13,
            horizontalalignment='left',
            verticalalignment='center')
        
    var_auc_list = []
    for pyrome_id in sorted(pyrome['PYROME']):
        var_auc_list += [np.sum(spread_season_dict[var][pyrome_id] - 1)/100]
    p = ax[1].barh(len(var_list) - i, np.mean(var_auc_list), color = color_sorted[::-1][i], height = 0.75)
    if np.mean(var_auc_list) > 0:
        ax[1].bar_label(p, labels=[f'{round(np.mean(var_auc_list),2)}'], label_type='edge',fontsize = 13, padding = 5)
    else:
        ax[1].text(0.01,len(var_list) - i,f'{round(np.mean(var_auc_list),2)}', fontsize = 13,
            horizontalalignment='left',
            verticalalignment='center')
        
    var_auc_list = []
    for pyrome_id in sorted(pyrome['PYROME']):
        var_auc_list += [np.sum(spread_anomaly_dict[var][pyrome_id] - 1)/100]
    p = ax[2].barh(len(var_list) - i, np.mean(var_auc_list), color = color_sorted[::-1][i], height = 0.75)
    ax[2].bar_label(p, labels=[f'{round(np.mean(var_auc_list),2)}'], label_type='edge',fontsize = 13, padding = 5)
    
    i += 1

for i in range(3):   
    ax[i].set_yticks(range(1,len(var_list)+1))
    ax[i].set_xlim([-0.15,0.65])
    ax[i].set_yticklabels([i.upper() for i in var_sorted], fontsize = 14)

    if i < 2:
        ax[i].tick_params(axis='y',which='both',direction = 'out',right=True, left=True)
ax[1].set_yticklabels([])
ax[2].yaxis.tick_right()
ax2 = ax[2].secondary_yaxis("left")
ax2.set_yticks(range(1,len(var_list)+1))
ax2.set_yticklabels([])
ax2.tick_params(axis="y", direction="out")

# ax[2].set_yticklabels([i.upper() for i in var_list][::-1])

ax[2].yaxis.tick_right()

for i in range(3):
    for ticklabel in ax[i].get_yticklabels():
        if ticklabel.get_text() in [j.upper() for j in ['e_s','rh','vpd','wspd','wspdmax','soil_m']]:
            ticklabel.set_color('blue')

    ax[i].set_xticks(np.arange(-0.1,0.61,0.1))
    ax[i].set_xticklabels([round(i,1) for i in np.arange(-0.1,0.61,0.1)],fontsize = 12)
    for threshold in np.arange(-0.1,0.61,0.1):
        ax[i].axvline(threshold, linestyle = '--', linewidth = 0.5, c= 'grey', zorder = 0)
    for threshold in np.arange(-0.1,0.61,0.1):
        ax[i].axvline(threshold, linestyle = '--', linewidth = 0.2, c= 'grey', zorder = 0)
    ax[i].axvline(0, linestyle = '-', linewidth = 1.5, c = 'black')
    ax[i].set_position([i*0.33,0,0.31,1])

ax[0].text(0.05,1.08,'a', fontsize = 26,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[0].transAxes)
ax[1].text(0.05,1.08,'b', fontsize = 26,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[1].transAxes)
ax[2].text(0.05,1.08,'c', fontsize = 26,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[2].transAxes)

ax[0].set_title('Daily Values', fontsize = 25, y = 1.03)
ax[1].set_title('Seasonality', fontsize = 25, y = 1.03)
ax[2].set_title('Anomalies', fontsize = 25, y = 1.03)

ax[1].set_xlabel('AUC score', fontsize = 18, labelpad = 10)


In [ ]:

colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive','teal', 'darkcyan', 'darkred', 'green', 'lightgray']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]

i = 0

spread_season_dict = {}
# count_dict = {}

for var in var_list:
    print(var)
    row, col = i //4, i % 4
    spread_season_dict[var] = pd.DataFrame()
#     count_dict[var] = pd.DataFrame()
    for pyrome_id in sorted(pyrome['PYROME']):
        spread_list = []
#         count_list = []
        data = FIRED[(FIRED['PYROME'] == pyrome_id)][['ig_date','event_day','date','dy_ar_km2','id']]
        # data = data[(data['dy_ar_km2'] >= 5)]
        data['fwi'] = era5_wrf_basin_season_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = data['date'].values).rank('day',pct = True)

        for threshold in np.arange(99,-1,-1):
            spread_list += [(np.mean(data[data['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(data['dy_ar_km2'])]
            # spread_list += [(np.mean(data[data['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(data['dy_ar_km2'])]
#             count_list += [len(data[data['fwi']*100 >= threshold])]
        
        spread_season_dict[var][pyrome_id] = spread_list
#         count_dict[var][pyrome_id] = count_list

    i += 1

In [ ]:

colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive','teal', 'darkcyan', 'darkred', 'green', 'lightgray']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]

i = 0

spread_anomaly_dict = {}
# count_dict = {}

for var in var_list:
    print(var)
    row, col = i //4, i % 4
    spread_anomaly_dict[var] = pd.DataFrame()
#     count_dict[var] = pd.DataFrame()
    for pyrome_id in sorted(pyrome['PYROME']):
        spread_list = []
#         count_list = []
        data = FIRED[(FIRED['PYROME'] == pyrome_id)][['ig_date','event_day','date','dy_ar_km2','id']]
        # data = data[(data['dy_ar_km2'] >= 5)]
        data['fwi'] = era5_wrf_basin_anomaly_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = data['date'].values).rank('day',pct = True)

        for threshold in np.arange(99,-1,-1):
            spread_list += [(np.mean(data[data['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(data['dy_ar_km2'])]
            # spread_list += [(np.mean(data[data['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(data['dy_ar_km2'])]
#             count_list += [len(data[data['fwi']*100 >= threshold])]
        
        spread_anomaly_dict[var][pyrome_id] = spread_list

    i += 1

In [ ]:
%%time

# var_list = ['e','e_s','rh','vpd','uv10','soil_m','hdw','haines','FFWI','kbdi','mffwi','fwi']
# var_list = ['e_s','rh','vpd','wspd','soil_m','hdw','haines','kbdi','ffwi','mffwi','fwi',"fm100","fm1000","erc","bi"]
colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive','teal', 'darkcyan', 'darkred', 'green', 'lightgray']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]

i = 0

spread_all_dict = {}
spread_forest_dict = {}
spread_savanna_dict = {}
spread_shrub_dict = {}
spread_grass_dict = {}
spread_non_forest_dict = {}
# count_dict = {}

spread_auc_df = pd.DataFrame()

for var in var_list:
    print(var)

    spread_all_dict[var] = pd.DataFrame()
    spread_forest_dict[var] = pd.DataFrame()
    spread_non_forest_dict[var] = pd.DataFrame()
    spread_savanna_dict[var] = pd.DataFrame()
    spread_shrub_dict[var] = pd.DataFrame()
    spread_grass_dict[var] = pd.DataFrame()
    
#     count_dict[var] = pd.DataFrame()
    data_all = pd.DataFrame()
    data_forest = pd.DataFrame()
    data_non_forest = pd.DataFrame()
    data_savanna = pd.DataFrame()
    data_shrub = pd.DataFrame()
    data_grass = pd.DataFrame()
    
    spread_all_list = []
    spread_forest_list = []
    spread_non_forest_list = []
    spread_savanna_list = []
    spread_shrub_list = []
    spread_grass_list = []
    
    for pyrome_id in sorted(pyrome['PYROME']):
        tmp_all_list = []
        tmp_forest_list = []
        tmp_non_forest_list = []
        tmp_savanna_list = []
        tmp_shrub_list = []
        tmp_grass_list = []
        
#         count_list = []
        tmp_all = FIRED[(FIRED['PYROME'] == pyrome_id)][['ig_date','event_day','date','dy_ar_km2','id']]
        tmp_forest = FIRED_forest[(FIRED_forest['PYROME'] == pyrome_id)][['ig_date','event_day','date','dy_ar_km2','id']]
        tmp_non_forest = FIRED_non_forest[(FIRED_non_forest['PYROME'] == pyrome_id)][['ig_date','event_day','date','dy_ar_km2','id']]
        # data = data[(data['event_day'] <= 3)]
        tmp_savanna = FIRED_savanna[(FIRED_savanna['PYROME'] == pyrome_id)][['ig_date','event_day','date','dy_ar_km2','id']]
        tmp_shrub = FIRED_shrub[(FIRED_shrub['PYROME'] == pyrome_id)][['ig_date','event_day','date','dy_ar_km2','id']]
        tmp_grass = FIRED_grass[(FIRED_grass['PYROME'] == pyrome_id)][['ig_date','event_day','date','dy_ar_km2','id']]
        # tmp_forest = tmp_forest[(tmp_forest['dy_ar_km2'] >= 5)]
        # tmp_non_forest = tmp_non_forest[(tmp_non_forest['dy_ar_km2'] >= 5)]
        
        tmp_all['fwi'] = era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = tmp_all['date'].values).rank('day',pct = True)
        tmp_forest['fwi'] = era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = tmp_forest['date'].values).rank('day',pct = True)
        tmp_non_forest['fwi'] = era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = tmp_non_forest['date'].values).rank('day',pct = True)
        tmp_savanna['fwi'] = era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = tmp_savanna['date'].values).rank('day',pct = True)
        tmp_shrub['fwi'] = era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = tmp_shrub['date'].values).rank('day',pct = True)
        tmp_grass['fwi'] = era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = tmp_grass['date'].values).rank('day',pct = True)
# data_forest = pd.concat([data_forest,tmp_forest])
        # data_non_forest = pd.concat([data_non_forest,tmp_non_forest])
        
        for threshold in np.arange(99,-1,-1):
            tmp_all_list += [(np.mean(tmp_all[tmp_all['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(tmp_all['dy_ar_km2'])]
            tmp_forest_list += [(np.mean(tmp_forest[tmp_forest['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(tmp_forest['dy_ar_km2'])]
            tmp_non_forest_list += [(np.mean(tmp_non_forest[tmp_non_forest['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(tmp_non_forest['dy_ar_km2'])]
            tmp_savanna_list += [(np.mean(tmp_savanna[tmp_savanna['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(tmp_savanna['dy_ar_km2'])]
            tmp_shrub_list += [(np.mean(tmp_shrub[tmp_shrub['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(tmp_shrub['dy_ar_km2'])]
            tmp_grass_list += [(np.mean(tmp_grass[tmp_grass['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(tmp_grass['dy_ar_km2'])]

        spread_all_dict[var][pyrome_id] = tmp_all_list
        spread_forest_dict[var][pyrome_id] = tmp_forest_list
        spread_non_forest_dict[var][pyrome_id] = tmp_non_forest_list
        spread_savanna_dict[var][pyrome_id] = tmp_savanna_list
        spread_shrub_dict[var][pyrome_id] = tmp_shrub_list
        spread_grass_dict[var][pyrome_id] = tmp_grass_list
    
        data_all = pd.concat([data_all, tmp_all])
        data_forest = pd.concat([data_forest, tmp_forest])
        data_non_forest = pd.concat([data_non_forest, tmp_non_forest])
        data_savanna = pd.concat([data_savanna, tmp_savanna])
        data_shrub = pd.concat([data_shrub, tmp_shrub])
        data_grass = pd.concat([data_grass, tmp_grass])
        
    for threshold in np.arange(99,-1,-1):
        spread_all_list += [(np.mean(data_all[data_all['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(data_all['dy_ar_km2'])]
        spread_forest_list += [(np.mean(data_forest[data_forest['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(data_forest['dy_ar_km2'])]
        spread_non_forest_list += [(np.mean(data_non_forest[data_non_forest['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(data_non_forest['dy_ar_km2'])]
        spread_savanna_list += [(np.mean(data_savanna[data_savanna['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(data_savanna['dy_ar_km2'])]
        spread_shrub_list += [(np.mean(data_shrub[data_shrub['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(data_shrub['dy_ar_km2'])]
        spread_grass_list += [(np.mean(data_grass[data_grass['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(data_grass['dy_ar_km2'])]
    
    spread_all_dict[var]['total'] = spread_all_list
    spread_forest_dict[var]['total'] = spread_forest_list
    spread_non_forest_dict[var]['total'] = spread_non_forest_list
    spread_savanna_dict[var]['total'] = spread_savanna_list
    spread_shrub_dict[var]['total'] = spread_shrub_list
    spread_grass_dict[var]['total'] = spread_grass_list
    
    i += 1
    

In [ ]:
%%time

# var_list = ['e','e_s','rh','vpd','uv10','soil_m','hdw','haines','FFWI','kbdi','mffwi','fwi']
# var_list = ['e_s','rh','vpd','wspd','soil_m','hdw','haines','kbdi','ffwi','mffwi','fwi',"fm100","fm1000","erc","bi"]
colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive','teal', 'darkcyan', 'darkred', 'green', 'lightgray']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]

FIRED_all = FIRED.copy()

spread_auc_dict = {}
for vegetation in ['all','forest','non_forest','savanna','shrub','grass']:

    print(vegetation)
    i = 0
    spread_auc_dict[vegetation] = {}
    for season_type in ['total','season','anomaly']:
        spread_auc_dict[vegetation][season_type] = pd.DataFrame()
        for var in var_list:
            # spread_auc_dict[vegetation][season_type][var] = pd.DataFrame()
            data = pd.DataFrame()
            
            
            for pyrome_id in sorted(pyrome['PYROME']):
                tmp_list = []
                tmp = eval(f'FIRED_{vegetation}')[(eval(f'FIRED_{vegetation}')['PYROME'] == pyrome_id)][['ig_date','event_day','date','dy_ar_km2','id']]
                if season_type == 'total': 
                    tmp['fwi'] = era5_wrf_basin_pct_rank_dict[var]['fwi'].sel(basin = pyrome_id).sel(day = tmp['date'].values).rank('day',pct = True)
                else: 
                    tmp['fwi'] = eval(f'era5_wrf_basin_{season_type}_pct_rank_dict')[var]['fwi'].sel(basin = pyrome_id).sel(day = tmp['date'].values).rank('day',pct = True)
                
                # for threshold in np.arange(99,-1,-1):
                #     tmp_list += [(np.mean(tmp[tmp['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(tmp['dy_ar_km2'])]
                # spread_auc_dict[vegetation][season_type][var][pyrome_id] = tmp_list
                
                data = pd.concat([data, tmp])
                
            spread_list = []
            for threshold in np.arange(99,-1,-1):
                spread_list += [(np.mean(data[data['fwi']*100 >= threshold]['dy_ar_km2']))/np.mean(data['dy_ar_km2'])]
                
            var_auc = [np.sum(spread_list)/100-1]
            
            spread_auc_dict[vegetation][season_type].loc['domain', var] = var_auc[0]
            
            i += 1

In [ ]:
fig,ax = plt.subplots(2,3,figsize = (16,11), dpi = 400, sharex = False, sharey = False)

# colors = ['violet','orangered','royalblue','orange','darkviolet','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive']
# var_list = ['e_s','rh','vpd','wspd','soil_m','hdw','haines','kbdi','ffwi','mffwi','fwi',"fm100","fm1000","erc","bi"]
colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive','lightgray', 'darkcyan', 'darkred', 'green']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]


ax[0,1].set_position([0.45,0.46,0.25,0.37])
ax[0,2].set_position([0.75,0.46,0.25,0.37])
ax[1,1].set_position([0.45,0.,0.25,0.37])
ax[1,2].set_position([0.75,0.,0.25,0.37])

ax[0,0].set_position([-0.02,0.46,0.38,0.35])
ax[1,0].set_position([-0.02,0.,0.38,0.35])

row = 0

for vegetation in ['forest','non_forest']:
    df_tmp = eval(f'FIRED_{vegetation}')
    spread_mean_auc_list = []
    
    # spread_forest_dict[vegetation] = {}
    # count_pdf_dict[vegetation] = {}
    
    i = 0
    # for var in var_list_short:
    for var in var_list:
        if i < 6:
            col = 1
        else:
            col = 2

        ax[row,col].plot(eval(f'spread_{vegetation}_dict')[var]['total'], color = colors[i], label = var.upper(),linewidth = 2.5)
        
        spread_mean_auc_list += [np.sum(eval(f'spread_{vegetation}_dict')[var]['total'])/(100) - 1]
        
        if row == 1:
            if col == 1:
                ax[row,col].legend(loc = 'upper right', fontsize = 16, handlelength = 1.3)
            else:
                ax[row,col].legend(loc = [1.05,0], fontsize = 20, handlelength = 1.3)

        i += 1
            
                
    auc_sorted = [i[0] for i in sorted(zip(spread_mean_auc_list,colors,var_list),key=lambda x:x[0])]
    color_sorted = [i[1] for i in sorted(zip(spread_mean_auc_list,colors,var_list),key=lambda x:x[0])]
    var_sorted = [i[2].upper() for i in sorted(zip(spread_mean_auc_list,colors,var_list),key=lambda x:x[0])]    

    # for threshold in np.arange(-0.2,0.61,0.1):
    #     ax[row,0].axhline(threshold, linestyle = '--', linewidth = 0.5, c= 'grey')
    ax[row,0].axhline(0, linestyle = '-', linewidth = 1.5, c = 'black')
    
    p = ax[row,0].bar(range(len(spread_mean_auc_list)),auc_sorted, color = color_sorted, alpha = 0.8)
    ax[row,0].bar_label(p, labels=[round(i,2) for i in auc_sorted], label_type='edge',fontsize = 12, rotation = -40, padding = 1)
    
    ax[row,0].set_xticks(range(len(spread_mean_auc_list)))
    ax[row,0].set_xticklabels(var_sorted,rotation = 70,fontsize = 16)
    
    ax[row,0].set_yticks(np.arange(-0.2,0.61,0.2))
    ax[row,0].set_yticklabels([round(i,1) for i in np.arange(-0.2,0.61,0.2)],fontsize = 17)
    ax[row,0].set_ylabel('AUC score',fontsize = 23)
    
    for threshold in np.arange(-0.1,1.0,0.1):
        if threshold == 0:
            ax[row,0].axhline(threshold, linewidth = 0.9, color = 'grey', linestyle = '-')
        else:
            ax[row,0].axhline(threshold, linewidth = 0.4, color = 'grey', linestyle = '--')

    for ticklabel in ax[row,0].get_xticklabels():
        if ticklabel.get_text() in [i.upper() for i in ['e_s','rh','vpd','wspd','wspdmax','soil_m']]:
            ticklabel.set_color('blue')
        if ticklabel.get_text() in [i.upper() for i in ['wspdmax']]:
            ticklabel.set_fontsize(15)
    
    # ax[row,0].tick_params(axis='x', colors=tick_color_list)
    ax[row,0].set_ylim([-0.18,0.91])
    ax[row,1].set_ylabel('Fire Spread\nRelative to Pyrome Mean', fontsize = 20, labelpad = 8)
  
    row += 1

for col in range(1,3):
    
    for row in range(2):
        ax[row,col].set_yticks(np.arange(0,9,1))
        ax[row,col].set_yticklabels([round(i,1) for i in np.arange(0,9,1)],fontsize = 17)

        ax[row,col].set_xticks(np.arange(0,101,20))
        ax[row,col].set_xticklabels([round(i,1) for i in np.arange(0,101,20)],fontsize = 17)

    for threshold in np.arange(0,9.1,1):
        if threshold == 1:
            ax[0,col].axhline(threshold, linewidth = 0.9, color = 'grey', linestyle = '-')
        else:
            ax[0,col].axhline(threshold, linewidth = 0.2, color = 'grey', linestyle = '--')
            
                
    ax[1,col].text(0.5,-0.15,'Index Percentile', fontsize = 21,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[1,col].transAxes)
    ax[0,col].set_ylim([-0.5,8])
    for row in range(1,2):
        ax[row,col].set_ylim([-0.25,4])
        for threshold in np.arange(0,4.1,0.5):
            if threshold == 1:
                ax[row,col].axhline(threshold, linewidth = 0.9, color = 'grey', linestyle = '-')
            else:
                ax[row,col].axhline(threshold, linewidth = 0.2, color = 'grey', linestyle = '--')

ax[0,0].text(-0.2,0.5,f'Forest\n(n = {len(FIRED_forest)}\nfire-days)', fontsize = 26,
            horizontalalignment='right',
            verticalalignment='center', transform=ax[0,0].transAxes)
ax[1,0].text(-0.2,0.5,f'Non-Forest\n(n = {len(FIRED_non_forest)}\nfire-days)', fontsize = 26,
            horizontalalignment='right',
            verticalalignment='center', transform=ax[1,0].transAxes)
    
ax[0,1].set_title('Hydrometeorological\nVariables', fontsize = 29, y = 1.05)
ax[0,2].set_title('Fire Indices', fontsize = 29, y = 1.05)

i = 0
letter_list = ['a','b','c','d','e','f','g','h','i','j','k','l','m']
for row in range(2):
    for col in range(3):
        ax[row,col].text(-0.09,1.10,letter_list[i], fontsize = 32,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[row,col].transAxes)
        i += 1


In [ ]:
fig,ax = plt.subplots(4,3,figsize = (16,20), dpi = 400, sharex = False, sharey = False)

colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive','lightgray', 'darkcyan', 'darkred', 'green']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]

ax[0,1].set_position([0.45,0.75,0.25,0.2])
ax[0,2].set_position([0.75,0.75,0.25,0.2])
ax[1,1].set_position([0.45,0.5,0.25,0.2])
ax[1,2].set_position([0.75,0.5,0.25,0.2])

ax[0,0].set_position([0.,0.75,0.36,0.2])
ax[1,0].set_position([0.,0.5,0.36,0.2])

ax[2,1].set_position([0.45,0.25,0.25,0.2])
ax[2,2].set_position([0.75,0.25,0.25,0.2])
ax[3,1].set_position([0.45,0,0.25,0.2])
ax[3,2].set_position([0.75,0.,0.25,0.2])

ax[2,0].set_position([0.,0.25,0.36,0.2])
ax[3,0].set_position([0.,0.,0.36,0.2])

row = 0

for vegetation in ['forest','savanna','shrub','grass']:
    df_tmp = eval(f'FIRED_{vegetation}')
    spread_mean_auc_list = []
    
    # spread_forest_dict[vegetation] = {}
    # count_pdf_dict[vegetation] = {}
    
    i = 0
    # for var in var_list_short:
    for var in var_list:
        if i < 5:
            col = 1
        else:
            col = 2

        ax[row,col].plot(eval(f'spread_{vegetation}_dict')[var]['total'], color = colors[i], label = var.upper(), linewidth = 2)
        
        spread_mean_auc_list += [np.sum(eval(f'spread_{vegetation}_dict')[var]['total'])/(100) - 1]
        
        if row == 3:
            if col == 1:
                ax[row,col].legend(loc = 'upper right', fontsize = 18, handlelength = 1.3)
            else:
                ax[row,col].legend(loc = [1.05,0], fontsize = 22, handlelength = 1.3)
            
        i += 1
            
                
    auc_sorted = [i[0] for i in sorted(zip(spread_mean_auc_list,colors,var_list),key=lambda x:x[0])]
    color_sorted = [i[1] for i in sorted(zip(spread_mean_auc_list,colors,var_list),key=lambda x:x[0])]
    var_sorted = [i[2].upper() for i in sorted(zip(spread_mean_auc_list,colors,var_list),key=lambda x:x[0])]    

    # for threshold in np.arange(-0.2,0.61,0.1):
    #     ax[row,0].axhline(threshold, linestyle = '--', linewidth = 0.5, c= 'grey')
    ax[row,0].axhline(0, linestyle = '-', linewidth = 1.5, c = 'black')
    
    p = ax[row,0].bar(range(len(spread_mean_auc_list)),auc_sorted, color = color_sorted, alpha = 0.8)
    ax[row,0].bar_label(p, labels=[round(i,2) for i in auc_sorted], label_type='edge',fontsize = 11, rotation = -40, padding = 3)
    
    ax[row,0].set_xticks(range(len(spread_mean_auc_list)))
    ax[row,0].set_xticklabels(var_sorted,rotation = 70,fontsize = 14)
    
    ax[row,0].set_yticks(np.arange(-0.2,0.81,0.2))
    ax[row,0].set_yticklabels([round(i,1) for i in np.arange(-0.2,0.81,0.2)],fontsize = 15)
    ax[row,0].set_ylabel('AUC score',fontsize = 18)
    
    for threshold in np.arange(-0.1,1.0,0.1):
        if threshold == 0:
            ax[row,0].axhline(threshold, linewidth = 0.9, color = 'grey', linestyle = '-')
        else:
            ax[row,0].axhline(threshold, linewidth = 0.4, color = 'grey', linestyle = '--')

    for ticklabel in ax[row,0].get_xticklabels():
        if ticklabel.get_text() in [i.upper() for i in ['e_s','rh','vpd','wspd','soil_m']]:
            ticklabel.set_color('blue')
    
    # ax[row,0].tick_params(axis='x', colors=tick_color_list)
    ax[row,0].set_ylim([-0.18,0.91])
    ax[row,1].set_ylabel('Fire Spread\nRelative to Pyrome Mean', fontsize = 17, labelpad = 8)
  
    row += 1

for col in range(1,3):
    for row in range(4):
        ax[row,col].set_yticks(np.arange(0,9,1))
        ax[row,col].set_yticklabels([round(i,1) for i in np.arange(0,9,1)],fontsize = 17)

        ax[row,col].set_xticks(np.arange(0,101,20))
        ax[row,col].set_xticklabels([round(i,1) for i in np.arange(0,101,20)],fontsize = 17)

    for threshold in np.arange(0,9.1,1):
        if threshold == 1:
            ax[0,col].axhline(threshold, linewidth = 0.9, color = 'grey', linestyle = '-')
        else:
            ax[0,col].axhline(threshold, linewidth = 0.2, color = 'grey', linestyle = '--')
            
                
    ax[3,col].text(0.5,-0.15,'Index Percentile', fontsize = 18,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[3,col].transAxes)
    ax[0,col].set_ylim([-0.5,8])
    for row in range(1,4):
        ax[row,col].set_ylim([-0.25,4])
        for threshold in np.arange(0,4.1,0.5):
            if threshold == 1:
                ax[row,col].axhline(threshold, linewidth = 0.9, color = 'grey', linestyle = '-')
            else:
                ax[row,col].axhline(threshold, linewidth = 0.2, color = 'grey', linestyle = '--')

ax[0,0].text(-0.2,0.5,f'Forest\n(n = {len(FIRED_forest)}\nfire-days)', fontsize = 26,
            horizontalalignment='right',
            verticalalignment='center', transform=ax[0,0].transAxes)
ax[1,0].text(-0.2,0.5,f'Savanna\n(n = {len(FIRED_savanna)}\nfire-days)', fontsize = 26,
            horizontalalignment='right',
            verticalalignment='center', transform=ax[1,0].transAxes)
ax[2,0].text(-0.2,0.5,f'Shrub\n(n = {len(FIRED_shrub)}\nfire-days)', fontsize = 26,
            horizontalalignment='right',
            verticalalignment='center', transform=ax[2,0].transAxes)
ax[3,0].text(-0.2,0.5,f'Grass\n(n = {len(FIRED_grass)}\nfire-days)', fontsize = 26,
            horizontalalignment='right',
            verticalalignment='center', transform=ax[3,0].transAxes)
    
ax[0,1].set_title('Hydrometeorological\nVariables', fontsize = 26, y = 1.02)
ax[0,2].set_title('Fire Indices', fontsize = 26, y = 1.02)

i = 0
letter_list = ['a','b','c','d','e','f','g','h','i','j','k','l','m']
for row in range(4):
    for col in range(3):
        ax[row,col].text(-0.09,1.10,letter_list[i], fontsize = 32,
            horizontalalignment='center',
            verticalalignment='center', transform=ax[row,col].transAxes)
        i += 1


In [ ]:
# fig,ax = plt.subplots(3,1,figsize = (8,9), dpi = 400, sharex = True)
fig,ax = plt.subplots(3,3,figsize = (16,16), dpi = 400, sharex = True)

colors = ['violet','orangered','royalblue','orange', 'peru','pink','magenta','olive','deepskyblue','goldenrod','darkviolet','forestgreen','blue','navy','limegreen', 'sienna','olive','lightgray', 'darkcyan', 'darkred', 'green']
var_list = ['e_s','rh','vpd','wspd','wspdmax','soil_m','hdw','haines','kbdi','ffwi','mffwi','isi','bui','dc','ffmc','dmc','fwi',"fm100","fm1000","erc","bi"]

row = 0
for vegetation in ['all','forest','non_forest']:
    mean_auc_list = [np.mean(spread_auc_dict[vegetation]['total'][var]) for var in var_list]

    auc_sorted = [i[0] for i in sorted(zip(mean_auc_list,colors,var_list),key=lambda x:x[0])][::-1]
    color_sorted = [i[1] for i in sorted(zip(mean_auc_list,colors,var_list),key=lambda x:x[0])][::-1]
    var_sorted = [i[2].upper() for i in sorted(zip(mean_auc_list,colors,var_list),key=lambda x:x[0])][::-1]

    i = 0
    for var in var_sorted:

        var = var.lower()
        p = ax[row,0].barh(len(var_list) - i, np.mean(spread_auc_dict[vegetation]['total'][var]), color = color_sorted[i], height = 0.75)
        if np.mean(spread_auc_dict[vegetation]['total'][var]) > 0:
            ax[row,0].bar_label(p, labels=[f'{round(np.mean(spread_auc_dict[vegetation]['total'][var]),2)}'], label_type='edge',fontsize = 13, padding = 5)
        else:
            ax[row,0].text(0.01,len(var_list) - i,f'{round(np.mean(spread_auc_dict[vegetation]['total'][var]),2)}', fontsize = 13,
                horizontalalignment='left',
                verticalalignment='center')
            
        p = ax[row,1].barh(len(var_list) - i, np.mean(spread_auc_dict[vegetation]['season'][var]), color = color_sorted[i], height = 0.75)
        if np.mean(spread_auc_dict[vegetation]['season'][var]) > 0:
            ax[row,1].bar_label(p, labels=[f'{round(np.mean(spread_auc_dict[vegetation]['season'][var]),2)}'], label_type='edge',fontsize = 13, padding = 5)
        else:
            ax[row,1].text(0.01,len(var_list) - i,f'{round(np.mean(spread_auc_dict[vegetation]['season'][var]),2)}', fontsize = 13,
                horizontalalignment='left',
                verticalalignment='center')
        
        p = ax[row,2].barh(len(var_list) - i, np.mean(spread_auc_dict[vegetation]['anomaly'][var]), color = color_sorted[i], height = 0.75)
        if np.mean(spread_auc_dict[vegetation]['anomaly'][var]) > 0:
            ax[row,2].bar_label(p, labels=[f'{round(np.mean(spread_auc_dict[vegetation]['anomaly'][var]),2)}'], label_type='edge',fontsize = 13, padding = 5)
        else:
            ax[row,2].text(0.01,len(var_list) - i,f'{round(np.mean(spread_auc_dict[vegetation]['anomaly'][var]),2)}', fontsize = 13,
                horizontalalignment='left',
                verticalalignment='center')
        i += 1

    for i in range(3):   
        ax[row,i].set_yticks(range(1,len(var_sorted)+1))
        ax[row,i].set_xlim([-0.25,0.85])
        ax[row,i].set_yticklabels([i.upper() for i in var_sorted][::-1], fontsize = 14)

        if i < 2:
            ax[row,i].tick_params(axis='y',which='both',direction = 'out',right=True, left=True)
    ax[row,1].set_yticklabels([])
    ax[row,2].yaxis.tick_right()
    ax2 = ax[row,2].secondary_yaxis("left")
    ax2.set_yticks(range(1,len(var_list)+1))
    ax2.set_yticklabels([])
    ax2.tick_params(axis="y", direction="out")

    # ax[2].set_yticklabels([i.upper() for i in var_list][::-1])

    ax[row,2].yaxis.tick_right()

    for i in range(3):
        for ticklabel in ax[row,i].get_yticklabels():
            if ticklabel.get_text() in [j.upper() for j in ['e_s','rh','vpd','wspd','soil_m']]:
                ticklabel.set_color('blue')

        ax[row,i].set_xticks(np.arange(-0.2,0.81,0.2))
        ax[row,i].set_xticklabels([round(i,1) for i in np.arange(-0.2,0.81,0.2)],fontsize = 12)
        for threshold in np.arange(-0.2,0.81,0.1):
            ax[row,i].axvline(threshold, linestyle = '--', linewidth = 0.5, c= 'grey', zorder = 0)
        for threshold in np.arange(-0.15,0.81,0.1):
            ax[row,i].axvline(threshold, linestyle = '--', linewidth = 0.2, c= 'grey', zorder = 0)
        ax[row,i].axvline(0, linestyle = '-', linewidth = 1.5, c = 'black')
        ax[row,i].set_position([i*0.33,1-(row+1)*0.33,0.31,0.26])


    row += 1

letter_list = ['a','b','c','d','e','f','g','h','i','j','k']
for i in range(9):
    row,col = i//3,i%3
    
    ax[row,col].text(0.05,1.08,letter_list[i], fontsize = 26,
                horizontalalignment='center',
                verticalalignment='center', transform=ax[row,col].transAxes)

ax[0,0].set_title('Daily Values', fontsize = 25, y = 1.04)
ax[0,1].set_title('Seasonality', fontsize = 25, y = 1.04)
ax[0,2].set_title('Anomalies', fontsize = 25, y = 1.04)

ax[0,0].text(-0.3,0.5,'All', fontsize = 24,
                horizontalalignment='right',
                verticalalignment='center', transform=ax[0,0].transAxes)
ax[1,0].text(-0.3,0.5,'Forest', fontsize = 24,
                horizontalalignment='right',
                verticalalignment='center', transform=ax[1,0].transAxes)
ax[2,0].text(-0.3,0.5,'Non-Forest', fontsize = 24,
                horizontalalignment='right',
                verticalalignment='center', transform=ax[2,0].transAxes)

ax[2,1].set_xlabel('AUC score', fontsize = 18, labelpad = 10)


In [ ]:
i = 1                                                                         
# best_var_list = ['erc','fm1000','bi','hdw','fwi','fm100','vpd']
# best_var_list = ['erc','fm1000','bi','hdw','fm100','fwi','vpd']
# best_var_list =['vpd','hdw','fwi','fm100','fm1000','erc','bi']
best_var_list =['hdw','dc','fwi','fm100','fm1000','erc','bi']

pyrome = gpd.read_file('data/Pyromes/Pyromes_lon105W.shp')
pyrome.index = pyrome['PYROME'].values

for var in best_var_list:
    pyrome[f'spread_{var}_total_auc'] = np.array(spread_dict[var].sum(axis = 0)/100-1)


##### Finding best variable for each pyrome
tmp = pyrome[[i for i in pyrome.columns if i.endswith('total_auc')]].idxmax(axis = 1,skipna=True)
tmp[:] = [i[7:-10] for i in tmp]

i = 1
for var in best_var_list:
    tmp[:] = np.where(tmp == var, i, tmp)
    i += 1
    
pyrome['spread_best_total_var'] = [int(i) for i in tmp]

for var in best_var_list:
    pyrome[f'spread_{var}_non_forest_auc'] = np.array(spread_non_forest_dict[var].iloc[:,:-1].sum(axis = 0)/100-1)
    pyrome[f'spread_{var}_forest_auc'] = np.array(spread_forest_dict[var].iloc[:,:-1].sum(axis = 0)/100-1)
        
    i += 1

##### Finding best variable for each pyrome
tmp = pyrome[[i for i in pyrome.columns if i.endswith('non_forest_auc')]].idxmax(axis = 1,skipna=True)
tmp[:] = [i[7:-15] for i in tmp]

i = 1
for var in best_var_list:
    tmp[:] = np.where(tmp == var, i, tmp)
    i += 1
    
pyrome['spread_best_non_forest_var'] = [int(i) for i in tmp]

### Forest only
tmp = pyrome[[i for i in pyrome.columns if (i.endswith('forest_auc') & ('non' not in i))]].idxmax(axis = 1,skipna=True)
tmp[:] = [i[7:-11] if type(i) is str else np.nan for i in tmp]

i = 1
for var in best_var_list:
    tmp[:] = np.where(tmp == var, (i), tmp)
    i += 1
    
pyrome['spread_best_forest_var'] = [int(i) if i > 0 else np.nan for i in tmp]
print('Done')

In [ ]:
fig,ax=plt.subplots(2,4,figsize=(16,9), dpi = 400, subplot_kw={'projection': ccrs.PlateCarree()})

i = 1
# best_var_list = ['erc','fm1000','bi','hdw','fwi','fm100','vpd']
# best_var_list = ['erc','fm1000','bi','hdw','fm100','fwi','vpd']
best_var_list =['hdw','dc','fwi','fm100','fm1000','erc','bi']

for var in best_var_list:

    row,col = i//4,i%4
    # if row == 1:
    #     col += 1

    bounds = list(np.arange(0.,1.21,0.1))
    cmap = cm.get_cmap('Reds', 12)
    colors = cmap(np.arange(0,cmap.N))
    norm = matplotlib.colors.BoundaryNorm(boundaries=bounds, ncolors = 12)

    pyrome.plot(ax = ax[row,col], column = f'spread_{var}_total_auc', cmap = cmap, norm = norm, edgecolor = 'k', linewidth = 0.8)
    # ax[row,col].pcolormesh(tree_cover.lon, tree_cover.lat, tree_cover[f'spread_{var}_forest_auc_by_pyrome'], cmap = cmap, norm = norm) #, alpha = tree_cover['tree_cover']/100
    # pyrome.plot(ax = ax[row,col], column = f'spread_{var}_non_forest_auc', cmap = cmap, norm = norm, facecolor = 'None', edgecolor = 'k', linewidth = 1)

    ax[row,col].set_title(var.upper(), fontsize = 20, y = 1.02)
    ax[row,col].set_position([col*0.23+0.075,0.5-row*0.5,0.2,0.45])

    ax[row,col].set_xlim([-125.7,-104.4])
    ax[row,col].set_ylim([30.2,50])
    
    i += 1
 
cax = fig.add_axes([0.99, 0.03, 0.03, 0.9])
sm = plt.cm.ScalarMappable(cmap = cmap, norm = norm)
sm._A = []
cbar = fig.colorbar(sm, cax=cax, extend = 'both')

cbar.set_ticks(np.arange(-0.,1.21,0.2))
cbar.set_ticklabels([round(i,2) for i in cbar.get_ticks()], fontsize = 14)
cbar.set_label('AUC score', rotation = 270, fontsize = 20, labelpad = 30)

###### Plotting for best variable in each pyrome
bounds = list(np.arange(0.5,len(best_var_list)+0.6,1))
# cmap = ListedColormap(['darkorchid','deepskyblue','mediumorchid','gold','dodgerblue','hotpink','palegoldenrod'])
cmap = ListedColormap(['gold','hotpink','pink','dodgerblue','deepskyblue','darkorchid','mediumorchid'])
# cmap = mpl.colors.LinearSegmentedColormap.from_list(
#     'Custom', cmap, 7)
# colors = cmap([0,1,2,4,5,6,7])
norm = matplotlib.colors.BoundaryNorm(boundaries=bounds, ncolors = len(best_var_list))

pyrome.plot(ax = ax[0,0], column = f'spread_best_total_var', cmap = cmap, norm = norm, vmin = 0.5, vmax = 7.5, edgecolor = 'k', linewidth = 0.8)
# ax[0,0].pcolormesh(tree_cover.lon, tree_cover.lat, tree_cover['spread_best_total_var'], cmap = cmap, norm = norm)
# pyrome.plot(ax = ax[0,0], column = f'spread_best_non_forest_var', facecolor = 'None', edgecolor = 'k', linewidth = 1)

cax = fig.add_axes([0.04, 0.53, 0.015, 0.4])
sm = plt.cm.ScalarMappable(cmap = cmap, norm = norm)
sm._A = []
cbar = fig.colorbar(sm, cax=cax)

cbar.set_ticks(np.arange(1,len(best_var_list)+0.1,1))
cbar.set_ticklabels([i.upper() for i in best_var_list], fontsize = 14)
cbar.ax.yaxis.set_ticks_position("left")
cbar.ax.yaxis.set_label_position("left")

ax[0,0].set_position([0.075,0.5,0.2,0.45])

ax[0,0].set_xlim([-125.7,-104.4])
ax[0,0].set_ylim([30.2,50])
    
ax[0,0].set_title('Best Variable for Pyrome', fontsize = 20, y = 1.02)

# plt.delaxes(ax[1,0])
